# FLIDS Master V3 — Resume & Recovery
Attach the Experiment 09 and Packages 10–13 results as ZIPs or extracted datasets. Attach optional CSVs when available. Run all cells. The recovery gate checks the historical files before training. To continue a later session, attach its V3 results archive. If several checkpoints are attached, select RESUME_ZIP.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess, importlib.util, hashlib
from IPython.display import display, FileLink
SMOKE_TEST = False
STAGES = [10,11,12,13]  # Completed NF bundles are reused.
SESSION_HOURS = 8.0
AUTO_RESUME = True
REQUIRE_ALL_INPUTS = False  # False = run available tracks, explicitly report blocked tracks.
PREFLIGHT_ONLY = False  # True = inspect inputs only, no model fit.
EXP09_ZIP = ''  # optional exact path to original ZIP OR extracted dataset root
RESUME_ZIP = ''  # optional ZIP OR extracted v2 results root
EDGE_CSV = ''  # autodetect official DNN-EdgeIIoT-dataset.csv
EDGE_FEATURES = []  # exact historical Safe-30 list, if known
EDGE_FEATURES_JSON = ''  # autodetect edge_safe30_features.json
ALLOW_NEW_EDGE_PROTOCOL25 = True  # explicit alternative to unavailable historical Safe-30; not the same schema
EDGE_CAP_PER_FAMILY = 6000
EXTERNAL_INPUTS = []  # autodetect FLIDS_external_inputs.json; verified 41-feature data only
TEMPORAL_INPUT = None  # autodetect NF-UNSW-NB15-v3.csv with verified timestamp column
WORK = (Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')) / ('flids_master_v3_smoke' if SMOKE_TEST else 'flids_master_v3_run')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True,exist_ok=True)
required=['numpy','pandas','torch','sklearn','scipy','lightgbm']
missing=[x for x in required if importlib.util.find_spec(x) is None]
assert not missing, f'Missing dependencies: {missing}'
print('WORK:',WORK)

In [ ]:
SOURCE = '"""Validate Kaggle datasets before any model fit. No data downloading or inferred harmonization."""\nfrom pathlib import Path\nimport hashlib, io, json, zipfile, csv, re\nimport numpy as np\nimport pandas as pd\n\nMEMBERS=(\'reproducibility/flids_exp02_cache.npz\',\'reproducibility/flids_exp02_fingerprints.npy\',\'reproducibility/data_preparation/dataset_audit.json\')\nDOMAINS={\'NF-UNSW-NB15\',\'NF-ToN-IoT\',\'NF-BoT-IoT\',\'NF-CSE-CIC-IDS2018\'}\n# New, predeclared protocol-field subset, not the historical Safe-30 list.\nEDGE_PROTOCOL25=[\'arp.opcode\',\'arp.hw.size\',\'icmp.unused\',\'http.content_length\',\'http.response\',\'tcp.connection.fin\',\'tcp.connection.rst\',\'tcp.connection.syn\',\'tcp.connection.synack\',\'tcp.flags.ack\',\'tcp.len\',\'udp.time_delta\',\'dns.qry.name.len\',\'dns.qry.qu\',\'dns.qry.type\',\'dns.retransmission\',\'dns.retransmit_request\',\'dns.retransmit_request_in\',\'mqtt.conflag.cleansess\',\'mqtt.len\',\'mqtt.msgtype\',\'mqtt.proto_len\',\'mqtt.topic_len\',\'mqtt.ver\',\'mbtcp.len\']\n\ndef pick(paths, label):\n    paths=sorted(set(Path(p).resolve() for p in paths))\n    if len(paths)>1: raise ValueError(f\'Multiple {label} matches; set explicit path: {paths}\')\n    return paths[0] if paths else None\n\ndef read_exp09(root, explicit=\'\'):\n    root=Path(root)\n    if explicit:\n        p=Path(explicit)\n        if not p.exists(): raise FileNotFoundError(p)\n        candidates=[p]\n    else:\n        candidates=list(root.rglob(\'FLIDS_Experiment09_Results.zip\'))\n        for p in root.rglob(\'flids_exp02_cache.npz\'):\n            if p.parent.name==\'reproducibility\' and all((p.parent.parent/n).is_file() for n in MEMBERS):candidates.append(p.parent.parent)\n    found=[]\n    for p in candidates:\n        if p.is_dir():\n            data={n:(p/n).read_bytes() for n in MEMBERS}\n        else:\n            with zipfile.ZipFile(p) as z:data={n:z.read(n) for n in MEMBERS}\n        h=hashlib.sha256()\n        for n in MEMBERS:h.update(n.encode());h.update(hashlib.sha256(data[n]).digest())\n        found.append((p,data,h.hexdigest()))\n    if not found:return None\n    if len({v[2] for v in found})!=1:raise ValueError(\'Different Exp09 inputs found; set EXP09_ZIP explicitly.\')\n    return found[0]\n\ndef validate_nf(root,work,features,explicit=\'\'):\n    found=read_exp09(root,explicit)\n    if found is None:raise FileNotFoundError(\'Attach FLIDS_Experiment09_Results as ZIP or extracted folders (including reproducibility/).\')\n    p,data,digest=found\n    audit=json.loads(data[MEMBERS[2]])\n    if audit.get(\'features\')!=features:raise ValueError(\'NF feature order differs from Experiment 02 engine.\')\n    if audit.get(\'source_rows_scanned\')!=75987976:raise ValueError(\'Unexpected original source row count; not the historical full Exp09 input.\')\n    with np.load(io.BytesIO(data[MEMBERS[0]]),allow_pickle=False) as a:\n        assert {\'x\',\'y\',\'d\',\'attack\'}<=set(a.files)\n        assert a[\'x\'].shape==(96000,41) and all(a[k].shape==(96000,) for k in (\'y\',\'d\',\'attack\'))\n        assert np.isfinite(a[\'x\']).all() and set(a[\'y\'])=={0,1} and set(a[\'d\'])==DOMAINS\n        for d in DOMAINS:\n            for y in (0,1):assert int(((a[\'d\']==d)&(a[\'y\']==y)).sum())==12000\n    fp=np.load(io.BytesIO(data[MEMBERS[1]]),allow_pickle=False);assert fp.shape[0]==96000\n    # Build a canonical minimal ZIP outside the work folder, so the existing engine can consume either input layout.\n    target=Path(work).parent/\'FLIDS_Exp09_Canonical_Input.zip\';tmp=target.with_suffix(\'.tmp\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_STORED) as z:\n        for n in MEMBERS:z.writestr(zipfile.ZipInfo(n,date_time=(2026,1,1,0,0,0)),data[n])\n    tmp.replace(target)\n    return str(target),digest,{\'state\':\'ready\',\'source\':str(p),\'samples\':96000,\'features\':41,\'domains\':sorted(DOMAINS),\'source_rows\':75987976,\'content_sha256\':digest}\n\ndef header(path):\n    return list(pd.read_csv(path,nrows=0).columns)\n\ndef check_csv(path,columns,numeric):\n    p=Path(path)\n    if not p.is_file():raise FileNotFoundError(p)\n    missing=set(columns)-set(header(p))\n    if missing:raise ValueError(f\'{p.name} missing columns: {sorted(missing)}\')\n    df=pd.read_csv(p,usecols=columns,nrows=2048,low_memory=False)\n    if df.empty:raise ValueError(f\'{p.name} empty\')\n    for col in numeric:pd.to_numeric(df[col],errors=\'raise\')\n    return str(p)\n\ndef resolve_manifest(root,manifest_name,entries):\n    p=pick(Path(root).rglob(manifest_name),manifest_name)\n    if p is None:return entries\n    if entries:raise ValueError(f\'Both manual and {manifest_name} supplied; use just one.\')\n    result=json.loads(p.read_text())\n    if not isinstance(result,list):raise ValueError(\'External manifest must contain a JSON list.\')\n    for e in result:\n        q=Path(e[\'path\']);e[\'path\']=str(q if q.is_absolute() else p.parent/q)\n    return result\n\ndef run_preflight(root,work,features,*,exp09=\'\',edge_csv=\'\',edge_features=None,edge_features_json=\'\',allow_new_edge=True,external_inputs=None,temporal_input=None,require_all=False):\n    root=Path(root);work=Path(work);report={};resolved={};external_inputs=external_inputs or []\n    try:\n        path,dig,info=validate_nf(root,work,features,exp09);report[\'NF_UQ\']=info;resolved.update(exp09_zip=path,nf_content_sha=dig)\n    except Exception as e:report[\'NF_UQ\']={\'state\':\'blocked\',\'reason\':str(e)}\n    try:\n        p=Path(edge_csv) if edge_csv else pick(root.rglob(\'DNN-EdgeIIoT-dataset.csv\'),\'DNN Edge CSV\')\n        if p is None:raise FileNotFoundError(\'Add official Edge-IIoTset; DNN-EdgeIIoT-dataset.csv required.\')\n        js=Path(edge_features_json) if edge_features_json else pick(root.rglob(\'edge_safe30_features.json\'),\'Safe-30 JSON\')\n        if edge_features and js:raise ValueError(\'Set either EDGE_FEATURES or Safe-30 JSON, not both.\')\n        fs=list(edge_features or (json.loads(js.read_text()) if js else []))\n        if fs:\n            if len(fs)!=30 or len(set(fs))!=30:raise ValueError(\'Original Safe-30 requires exactly 30 unique names.\')\n            schema=\'Safe-30\'\n        elif allow_new_edge:fs=[c for c in EDGE_PROTOCOL25 if not c.startswith(\'dns.\')];schema=\'Edge-Numeric19-v3\'\n        else:raise ValueError(\'Missing audited original edge_safe30_features.json.\')\n        check_csv(p,fs+[\'Attack_label\',\'Attack_type\'],[\'Attack_label\'])\n        report[\'Edge\']={\'state\':\'ready_schema_check\',\'source\':str(p),\'schema\':schema,\'features\':fs,\'note\':\'Full numeric and 14-family validation during preparation; new subset is not historical Safe-30.\'};resolved.update(edge_csv=str(p),edge_features=fs,edge_schema=schema)\n    except Exception as e:report[\'Edge\']={\'state\':\'blocked\',\'reason\':str(e)}\n    try:\n        entries=resolve_manifest(root,\'FLIDS_external_inputs.json\',external_inputs)\n        if not entries:raise ValueError(\'Add verified harmonized independent external domain CSV(s) and FLIDS_external_inputs.json; raw CICIoT2023/X-IIoTID cannot be used directly.\')\n        names=[]\n        for e in entries:\n            if not e.get(\'schema_verified\') or not e.get(\'provenance\'):raise ValueError(\'Each external domain requires schema_verified=true and provenance of equivalent NetFlow feature extraction.\')\n            name=e[\'domain\'];names.append(name)\n            if name in DOMAINS or \'v3\' in name.lower() and any(t in name.lower() for t in [\'unsw\',\'ton\',\'bot\',\'cicids2018\']):raise ValueError(\'NF-v3 versions of original domains are not independent external domains.\')\n            check_csv(e[\'path\'],features+[e.get(\'label\',\'Label\'),e.get(\'family\',\'Attack\')],features+[e.get(\'label\',\'Label\')])\n        assert len(set(names))==len(names)\n        report[\'External\']={\'state\':\'ready_schema_check\',\'domains\':names};resolved[\'external_inputs\']=entries\n    except Exception as e:report[\'External\']={\'state\':\'blocked\',\'reason\':str(e)}\n    try:\n        e=dict(temporal_input) if temporal_input else None\n        if e is None:\n            matches=list(root.rglob(\'NF-UNSW-NB15-v3.csv\'))\n            if not matches:\n                for candidate in root.rglob(\'*.csv\'):\n                    if \'nfunswnb15v3\' in re.sub(r\'[^a-z0-9]\',\'\',str(candidate.relative_to(root)).lower()):\n                        try:\n                            if set(features+[\'FLOW_START_MILLISECONDS\',\'Label\',\'Attack\'])<=set(header(candidate)):matches.append(candidate)\n                        except Exception:pass\n            p=pick(matches,\'official NF-UNSW-NB15-v3 CSV\')\n            if p is None:raise FileNotFoundError(\'Add official NF-UNSW-NB15-v3.csv (timestamped) or configure TEMPORAL_INPUT.\')\n            e={\'path\':str(p),\'domain\':\'NF-UNSW-NB15-v3-temporal\',\'time_col\':\'FLOW_START_MILLISECONDS\',\'time_unit\':\'ms\',\'label\':\'Label\',\'family\':\'Attack\',\'group_col\':None,\'cap_per_class_block\':12000}\n        cols=features+[e.get(\'label\',\'Label\'),e.get(\'family\',\'Attack\'),e[\'time_col\']]+([e[\'group_col\']] if e.get(\'group_col\') else [])\n        check_csv(e[\'path\'],cols,[e.get(\'label\',\'Label\')]+([e[\'time_col\']] if e.get(\'time_unit\') else []))\n        report[\'Temporal\']={\'state\':\'ready_schema_check\',\'source\':e[\'path\'],\'time_col\':e[\'time_col\'],\'time_unit\':e.get(\'time_unit\'),\'capture_group\':e.get(\'group_col\'),\'note\':\'Chronological block validity checked on full data later; not an independent new domain.\'};resolved[\'temporal_input\']=e\n    except Exception as e:report[\'Temporal\']={\'state\':\'blocked\',\'reason\':str(e)}\n    report[\'Stages\']={\'10\':\'NF_UQ\',\'11\':\'NF_UQ\',\'12\':\'NF_UQ\',\'13\':[\'NF_UQ (leave-two)\',\'Edge\',\'External\',\'Temporal\']}\n    report[\'all_inputs_ready\']=all(report[k][\'state\'].startswith(\'ready\') for k in (\'NF_UQ\',\'Edge\',\'External\',\'Temporal\'))\n    (work/\'input_preflight.json\').write_text(json.dumps(report,indent=2));print(json.dumps(report,indent=2))\n    if report[\'NF_UQ\'][\'state\']!=\'ready\':raise RuntimeError(\'NF-UQ input invalid. See input_preflight.json.\')\n    if require_all and not report[\'all_inputs_ready\']:raise RuntimeError(\'Full run blocked by missing/invalid inputs; see input_preflight.json. No model fits started. Set REQUIRE_ALL_INPUTS=False only for partial execution.\')\n    return resolved,report\n'
(WORK/'preflight.py').write_text(SOURCE,encoding="utf-8")


In [ ]:
SOURCE = '"""Experiment 02: strictly source-held-out NetFlow V2 FL-IDS benchmark.\nEnglish-only; no target-domain tuning or target examples in source training.\nFive methods, four independent source dataset holdouts, five seeds, low-FPR evaluation.\n"""\nimport os\nfor k in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\'):os.environ[k]=\'1\'\nimport argparse,csv,hashlib,json,time,traceback,zipfile,faulthandler,random\nfrom pathlib import Path\nfrom collections import Counter,defaultdict\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom sklearn.metrics import roc_auc_score,average_precision_score\nfaulthandler.enable(all_threads=True)\nROOT=Path(os.getenv(\'FLIDS_EXP02_ROOT\',\'/kaggle/working\' if Path(\'/kaggle/working\').exists() else \'/mnt/data/exp02_local\'))\nOUT=ROOT/\'FLIDS_Experiment02_Results\';OUT.mkdir(parents=True,exist_ok=True)\nCACHE=ROOT/\'flids_exp02_cache.npz\'\nDOMAINS=[\'NF-UNSW-NB15\',\'NF-ToN-IoT\',\'NF-BoT-IoT\',\'NF-CSE-CIC-IDS2018\']\nFEATURES=[\'L4_SRC_PORT\',\'L4_DST_PORT\',\'PROTOCOL\',\'L7_PROTO\',\'IN_BYTES\',\'OUT_BYTES\',\'IN_PKTS\',\'OUT_PKTS\',\'FLOW_DURATION_MILLISECONDS\',\'TCP_FLAGS\',\'CLIENT_TCP_FLAGS\',\'SERVER_TCP_FLAGS\',\'DURATION_IN\',\'DURATION_OUT\',\'MIN_TTL\',\'MAX_TTL\',\'LONGEST_FLOW_PKT\',\'SHORTEST_FLOW_PKT\',\'MIN_IP_PKT_LEN\',\'MAX_IP_PKT_LEN\',\'SRC_TO_DST_SECOND_BYTES\',\'DST_TO_SRC_SECOND_BYTES\',\'RETRANSMITTED_IN_BYTES\',\'RETRANSMITTED_IN_PKTS\',\'RETRANSMITTED_OUT_BYTES\',\'RETRANSMITTED_OUT_PKTS\',\'SRC_TO_DST_AVG_THROUGHPUT\',\'DST_TO_SRC_AVG_THROUGHPUT\',\'NUM_PKTS_UP_TO_128_BYTES\',\'NUM_PKTS_128_TO_256_BYTES\',\'NUM_PKTS_256_TO_512_BYTES\',\'NUM_PKTS_512_TO_1024_BYTES\',\'NUM_PKTS_1024_TO_1514_BYTES\',\'TCP_WIN_MAX_IN\',\'TCP_WIN_MAX_OUT\',\'ICMP_TYPE\',\'ICMP_IPV4_TYPE\',\'DNS_QUERY_ID\',\'DNS_QUERY_TYPE\',\'DNS_TTL_ANSWER\',\'FTP_COMMAND_RET_CODE\']\nMETHODS=[\'PooledMLP\',\'PooledLightGBM\',\'LocalMLP\',\'FedAvgMLP\',\'FedProxMLP\']\nCFG=dict(seeds=[0,42,3407,123,2026],domains=DOMAINS,methods=METHODS,features=FEATURES,source_clients=3,rounds=30,local_epochs=1,batch_size=256,client_round_cap=512,lr=.001,prox_mu=.01,target_fprs=[.001,.01,.05],minimum_benign_for_001=1000,max_rows_per_domain_class=12000,source_validation_fraction=.20,lightgbm_trees=130,archive=\'FLIDS_Experiment02_Results.zip\',schema=\'UQ NetFlow V2; do not substitute Edge-IIoTset\',risk_notes=[\'Three different source datasets are FL clients, not actual organizational deployments\',\'Stratified reservoir changes prevalence and reported precision/AUPRC\',\'No target-domain examples or labels used in training, scaling, model selection, or thresholds\',\'Source-only validation thresholds do not guarantee the same target FPR\',\'LocalMLP means mean score ensemble of three separately trained domain models\',\'Full numeric feature fingerprints are a row-overlap proxy, not session/capture independence\',\'Centralized baselines require data pooling; recorded FL bytes omit network overhead\'])\nif os.getenv(\'FLIDS_EXP02_SMOKE\')==\'1\':CFG.update(seeds=[0],domains=DOMAINS[:2],rounds=2,client_round_cap=128,max_rows_per_domain_class=220,lightgbm_trees=18,minimum_benign_for_001=1000)\nif os.getenv(\'FLIDS_EXP02_SEEDS\'):CFG[\'seeds\']=[int(z) for z in os.environ[\'FLIDS_EXP02_SEEDS\'].split(\',\')]\n\ndef save_json(path,obj):Path(path).write_text(json.dumps(obj,indent=2,default=str),encoding=\'utf-8\')\ndef safe_header(path):\n with open(path,\'r\',encoding=\'utf-8-sig\',errors=\'replace\',newline=\'\') as f:return next(csv.reader(f))\ndef normalized(t):return \'\'.join(c for c in str(t).lower() if c.isalnum())\ndef canonical_domain(t):\n s=normalized(t)\n if \'unsw\' in s:return DOMAINS[0]\n if \'ton\' in s and \'iot\' in s:return DOMAINS[1]\n if \'bot\' in s and \'iot\' in s:return DOMAINS[2]\n if \'cse\' in s or \'cicids2018\' in s:return DOMAINS[3]\n return None\n\ndef discover():\n roots=[Path(\'/kaggle/input\'),ROOT/\'input\']\n if os.getenv(\'NF_UQ_DATA\'):roots.insert(0,Path(os.environ[\'NF_UQ_DATA\']))\n found=[];diagnostics=[]\n for root in roots:\n  if not root.exists():continue\n  paths=[root] if root.is_file() else list(root.rglob(\'*.csv\'))\n  for p in paths:\n   if \'v3\' in normalized(p.name):continue\n   try:\n    head=safe_header(p);cols=set(head);has_features=set(FEATURES).issubset(cols)\n    has_label=bool({\'Label\',\'label\'}&cols)\n    has_domain=\'Dataset\' in cols or canonical_domain(p.name) is not None\n    diagnostics.append(dict(path=str(p),columns=len(cols),valid_features=has_features,valid_label=has_label,domain_source=has_domain,missing_features=sorted(set(FEATURES)-cols)[:12]))\n    if has_features and has_label and has_domain:found.append((p,head))\n   except Exception as e:diagnostics.append(dict(path=str(p),error=str(e)[:180]))\n save_json(OUT/\'input_discovery.json\',dict(files=diagnostics,valid_candidates=[str(p) for p,_ in found]))\n combined=[v for v in found if normalized(v[0].name).startswith(\'nfuqnidsv2\')]\n if combined:return combined[:1]\n selected=[];used=set()\n for p,h in sorted(found,key=lambda v:str(v[0])):\n  domain=canonical_domain(p.name)\n  if domain and domain not in used:selected.append((p,h));used.add(domain)\n if len(selected)==4:return selected\n if found and any(\'Dataset\' in h for _,h in found):return [next(v for v in found if \'Dataset\' in v[1])]\n return selected\n\ndef synthetic_fixture():\n p=ROOT/\'synthetic_nf_exp02_SOFTWARE_TEST_ONLY.csv\';ROOT.mkdir(parents=True,exist_ok=True)\n rng=np.random.default_rng(51)\n with p.open(\'w\',newline=\'\') as f:\n  w=csv.writer(f);w.writerow(FEATURES+[\'Label\',\'Attack\',\'Dataset\'])\n  for d,domain in enumerate(DOMAINS):\n   for y in (0,1):\n    for i in range(260):\n     a=rng.gamma(2,1,len(FEATURES));a[0]=100+d*35+y*4;a[1]=200+d*11;a[4]+=y*(1.4+d*.2);a[5]+=y*.6;a[6]+=y*.8\n     w.writerow([f\'{v:.6f}\' for v in a]+[y,\'Attack\' if y else \'Benign\',domain+\'-v2\'])\n return p\n\ndef prepare():\n save_json(OUT/\'experiment_config.json\',CFG)\n if os.getenv(\'FLIDS_EXP02_SMOKE\')==\'1\':p=synthetic_fixture();found=[(p,safe_header(p))]\n else:found=discover()\n if not found:\n  raise FileNotFoundError(\'NF_UQ_NETFLOW_V2_NOT_FOUND: Attach the combined NF-UQ-NIDS-v2.csv or all four NF-v2 files in Kaggle Add Input; see input_discovery.json. Edge-IIoTset cannot be substituted. Set NF_UQ_DATA to a local CSV path if needed.\')\n rng=np.random.default_rng(2077);reservoir=defaultdict(list);seen=Counter();invalid=Counter();files_report=[]\n csv.field_size_limit(10_000_000)\n for path,head in found:\n  col={name:i for i,name in enumerate(head)}\n  lcol=col.get(\'Label\',col.get(\'label\'))\n  dcol=col.get(\'Dataset\');acol=col.get(\'Attack\')\n  fixed=canonical_domain(path.name)\n  good=bad=0\n  with path.open(\'r\',encoding=\'utf-8-sig\',errors=\'replace\',newline=\'\',buffering=1024*1024) as f:\n   reader=csv.reader(f);next(reader)\n   for row in reader:\n    if len(row)!=len(head):bad+=1;continue\n    domain=canonical_domain(row[dcol]) if dcol is not None else fixed\n    if domain not in DOMAINS:invalid[\'unknown_domain\']+=1;continue\n    label=normalized(row[lcol])\n    if label in (\'0\',\'normal\',\'benign\'):y=0\n    elif label in (\'1\',\'attack\',\'malicious\'):y=1\n    else:invalid[\'invalid_binary_label\']+=1;continue\n    try:\n     vals=np.array([float(row[col[c]]) if row[col[c]].strip() else 0. for c in FEATURES],dtype=np.float32)\n     if not np.isfinite(vals).all():invalid[\'nonfinite_values_replaced\']+=1;vals=np.nan_to_num(vals,nan=0.,posinf=0.,neginf=0.)\n    except Exception:bad+=1;continue\n    key=(domain,y);seen[key]+=1;n=seen[key]\n    item=(vals,str(row[acol]) if acol is not None else \'\')\n    buf=reservoir[key]\n    if len(buf)<CFG[\'max_rows_per_domain_class\']:buf.append(item)\n    else:\n     j=int(rng.integers(0,n))\n     if j<len(buf):buf[j]=item\n    good+=1\n   \n  files_report.append(dict(path=str(path),valid_rows=good,malformed_or_invalid_numeric=bad))\n  print(\'SCANNED\',path.name,good,\'invalid\',bad,flush=True)\n x=[];y=[];d=[];attack=[]\n for (domain,label),rows in sorted(reservoir.items()):\n  for a,t in rows:x.append(a);y.append(label);d.append(domain);attack.append(t)\n if not x:raise RuntimeError(\'NO_VALID_SOURCE_ROWS\')\n x=np.asarray(x,dtype=np.float32);y=np.asarray(y,dtype=np.int8);d=np.asarray(d);attack=np.asarray(attack)\n cov={f\'{domain}|{label}\':int(( (d==domain)&(y==label)).sum()) for domain in DOMAINS for label in (0,1)}\n min_count=100 if os.getenv(\'FLIDS_EXP02_SMOKE\')!=\'1\' else 50\n lacking=[k for k,v in cov.items() if v<min_count]\n save_json(OUT/\'data_coverage.json\',dict(pooled_rows=len(x),domain_class_counts=cov,minimum_samples_per_class=min_count,valid=not lacking,missing_or_small=lacking))\n if lacking:raise RuntimeError(\'DOMAIN_CLASS_COVERAGE_FAILED \'+str(lacking))\n np.savez_compressed(CACHE,x=x,y=y,d=d,attack=attack)\n # Exact numeric feature rows are used only as a *proxy* for duplicate overlap.\n fingerprints=np.asarray([hashlib.blake2b(np.ascontiguousarray(row).tobytes(),digest_size=12).hexdigest() for row in x])\n np.save(ROOT/\'flids_exp02_fingerprints.npy\',fingerprints)\n collision_groups=0;cross_domain_groups=0\n uniq,inv=np.unique(fingerprints,return_inverse=True)\n groups=defaultdict(set)\n for idx,k in enumerate(inv):groups[int(k)].add(str(d[idx]))\n cross_domain_groups=sum(len(v)>1 for v in groups.values())\n save_json(OUT/\'dataset_audit.json\',dict(original_counts={f\'{a}|{b}\':int(v) for (a,b),v in seen.items()},sample_counts=cov,source_files=files_report,invalid=dict(invalid),source_rows_scanned=sum(r[\'valid_rows\'] for r in files_report),sampled_rows=len(x),unique_numeric_fingerprints=len(uniq),duplicate_extra_rows=len(x)-len(uniq),cross_domain_fingerprint_groups=cross_domain_groups,duplicate_scope=\'capped sample, not full-source dedup\',split_independence=\'source dataset + exact feature fingerprint grouping for source train/validation; not true session-level independence\',features=FEATURES,sampling=\'uniform per domain and binary class; prevalence intentionally altered\',source_domain_definition=\'four separately collected sources within common NF-UQ NetFlow-v2 schema\'))\n manifests=[]\n for held in DOMAINS:\n  for seed in CFG[\'seeds\']:\n   tr,va,te=split_indices(y,d,fingerprints,held,seed)\n   for role,ids in [(\'train\',tr),(\'validation\',va),(\'test\',te)]:\n    manifests.append(dict(holdout=held,seed=seed,role=role,n=len(ids),benign=int(np.sum(y[ids]==0)),attack=int(np.sum(y[ids]==1)),index_digest=digest(ids),source_domains=\';\'.join(sorted(set(d[ids])))))\n pd.DataFrame(manifests).to_csv(OUT/\'split_manifest.csv\',index=False)\n print(\'EXP02_PREPARED\',len(x),\'rows, domain counts\',cov,\'folds\',len(CFG[\'seeds\'])*len(DOMAINS),flush=True)\n\ndef digest(ids):return hashlib.sha256(np.sort(np.asarray(ids,dtype=np.int64)).tobytes()).hexdigest()\ndef split_indices(y,d,fp,held,seed):\n tr=[];va=[];te=np.flatnonzero(d==held)\n for domain in DOMAINS:\n  if domain==held:continue\n  ids=np.flatnonzero(d==domain)\n  groups=fp[ids];unique=np.unique(groups)\n  if len(unique)<10:raise RuntimeError(f\'INSUFFICIENT_UNIQUE_GROUPS {domain}\')\n  a,b=next(GroupShuffleSplit(n_splits=1,test_size=CFG[\'source_validation_fraction\'],random_state=seed).split(ids,y[ids],groups))\n  tr.extend(ids[a]);va.extend(ids[b])\n tr=np.asarray(tr,dtype=np.int64);va=np.asarray(va,dtype=np.int64)\n assert not np.intersect1d(tr,va).size and not np.intersect1d(tr,te).size\n assert set(d[tr])==set(DOMAINS)-{held} and set(d[te])=={held}\n # Numeric fingerprint group overlap across *source* domains may occur; report, do not hide.\n for role,ids in [(\'train\',tr),(\'validation\',va),(\'test\',te)]:\n  if len(set(y[ids]))!=2:raise RuntimeError(f\'MISSING_BINARY_CLASS {held} {seed} {role}\')\n return tr,va,te\n\ndef transform(raw):\n # Fixed, source-agnostic signed log transform; no statistics or target data fitted.\n x=np.asarray(raw,dtype=np.float64)\n return np.clip(np.sign(x)*np.log1p(np.abs(np.clip(x,-1e12,1e12))),-18,18).astype(np.float32)\n\ndef threshold(scores,target):\n scores=np.asarray(scores,dtype=np.float64)\n if not scores.size or not np.isfinite(scores).all():raise RuntimeError(\'BAD_BENIGN_VALIDATION_SCORES\')\n order=np.sort(scores);k=int(np.floor(target*len(scores)))\n pivot=order[-k] if k else order[-1]\n t=float(np.nextafter(pivot,np.inf))\n actual=float(np.mean(scores>=t))\n if actual>target+1e-12:raise RuntimeError(\'THRESHOLD_INVARIANT_BROKEN\')\n return t,actual\n\ndef seedall(s):random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.set_num_threads(1)\nclass MLP(nn.Module):\n def __init__(self,n):super().__init__();self.net=nn.Sequential(nn.Linear(n,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU(),nn.Linear(32,1))\n def forward(self,x):return self.net(x).squeeze(-1)\n\ndef local_train(initial,x,y,idx,seed,steps,device,prox=False,global_state=None):\n model=MLP(x.shape[1]);model.load_state_dict(initial);model.to(device);model.train()\n opt=torch.optim.Adam(model.parameters(),lr=CFG[\'lr\'])\n rng=np.random.default_rng(seed);global_weights=[w.detach().to(device) for w in initial.values()]\n for _ in range(steps):\n  batch=rng.choice(idx,size=min(CFG[\'batch_size\'],len(idx)),replace=len(idx)<CFG[\'batch_size\'])\n  xb=torch.as_tensor(x[batch],device=device);yb=torch.as_tensor(y[batch],device=device,dtype=torch.float32)\n  opt.zero_grad(set_to_none=True)\n  loss=F.binary_cross_entropy_with_logits(model(xb),yb)\n  if prox:\n   penalty=sum((p-g).square().sum() for p,g in zip(model.parameters(),global_weights))\n   loss=loss+CFG[\'prox_mu\']*.5*penalty\n  loss.backward();opt.step()\n return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}\n\n@torch.no_grad()\ndef predict_model(state,x,idx,device):\n m=MLP(x.shape[1]);m.load_state_dict(state);m.to(device).eval();scores=[]\n for chunk in np.array_split(idx,max(1,int(np.ceil(len(idx)/4096)))):\n  if len(chunk):scores.append(torch.sigmoid(m(torch.as_tensor(x[chunk],device=device))).cpu().numpy())\n return np.concatenate(scores).astype(np.float64)\n\ndef fit_method(method,x,y,d,train,val,test,seed,rounds,device):\n tic=time.perf_counter();communication=0;steps=0;local_stats=[]\n if method==\'PooledLightGBM\':\n  from lightgbm import LGBMClassifier\n  model=LGBMClassifier(n_estimators=CFG[\'lightgbm_trees\'],num_leaves=15,max_depth=5,learning_rate=.05,min_child_samples=15,reg_lambda=1,verbosity=-1,n_jobs=1,random_state=seed)\n  model.fit(x[train],y[train]);sv=model.predict_proba(x[val])[:,1];st=model.predict_proba(x[test])[:,1];steps=CFG[\'lightgbm_trees\']\n else:\n  seedall(seed);m=MLP(x.shape[1]);base={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}\n  params=sum(w.numel() for w in m.parameters())\n  clients=[train[d[train]==domain] for domain in DOMAINS if (d[train]==domain).any()]\n  if len(clients)!=3 or min(map(len,clients))<10:raise RuntimeError(\'CLIENT_DOMAIN_ALLOCATION_INVALID\')\n  step_per_client=[max(1,int(np.ceil(min(len(ids),CFG[\'client_round_cap\'])/CFG[\'batch_size\'])))*CFG[\'local_epochs\'] for ids in clients]\n  if method==\'PooledMLP\':\n   s=base\n   for r in range(rounds):\n    for c,stepcount in enumerate(step_per_client):\n     s=local_train(s,x,y,train,seed+r*300+c*79,stepcount,device)\n     steps+=stepcount\n   sv=predict_model(s,x,val,device);st=predict_model(s,x,test,device)\n  elif method==\'LocalMLP\':\n   vals=[];tests=[]\n   for c,ids in enumerate(clients):\n    s=base\n    for r in range(rounds):\n     s=local_train(s,x,y,ids,seed+r*300+c*79,step_per_client[c],device)\n     steps+=step_per_client[c]\n    pv=predict_model(s,x,val,device);pt=predict_model(s,x,test,device)\n    vals.append(pv);tests.append(pt)\n    local_stats.append(dict(client=c,source_domain=str(d[ids[0]]),train_samples=len(ids),auroc=float(roc_auc_score(y[test],pt)),auprc=float(average_precision_score(y[test],pt))))\n   sv=np.mean(vals,axis=0);st=np.mean(tests,axis=0)\n  else:\n   s=base\n   for r in range(rounds):\n    states=[];sizes=[]\n    for c,ids in enumerate(clients):\n     states.append(local_train(s,x,y,ids,seed+r*300+c*79,step_per_client[c],device,prox=(method==\'FedProxMLP\')))\n     sizes.append(len(ids));steps+=step_per_client[c]\n    weights=np.asarray(sizes,dtype=float);weights/=weights.sum()\n    s={name:sum(item[name]*float(w) for item,w in zip(states,weights)) for name in s}\n    communication+=2*len(clients)*params*4\n   sv=predict_model(s,x,val,device);st=predict_model(s,x,test,device)\n return np.asarray(sv,dtype=np.float64),np.asarray(st,dtype=np.float64),dict(train_seconds=time.perf_counter()-tic,optimizer_steps=steps,communication_bytes=communication,local_clients=local_stats)\n\ndef tasks(pilot=False):\n ds=CFG[\'domains\'][:1] if pilot else DOMAINS\n ss=CFG[\'seeds\'][:1] if pilot else CFG[\'seeds\']\n ms=[\'PooledLightGBM\',\'FedAvgMLP\'] if pilot else METHODS\n return [(a,b,c) for a in ds for b in ss for c in ms]\n\ndef worker(mode,worker_id,nworkers):\n data=np.load(CACHE);raw=data[\'x\'];x=transform(raw);y=data[\'y\'];d=data[\'d\'];fp=np.load(ROOT/\'flids_exp02_fingerprints.npy\')\n pending=[task for i,task in enumerate(tasks(mode==\'pilot\')) if i%nworkers==worker_id]\n errors=[];device=f\'cuda:{worker_id%torch.cuda.device_count()}\' if torch.cuda.is_available() else \'cpu\'\n for ix,(hold,seed,method) in enumerate(pending,1):\n  key=f\'{mode}__{normalized(hold)}__{seed}__{method}\'\n  result_path=OUT/\'records\'/f\'{key}.json\';pred_path=OUT/\'predictions_raw\'/f\'{key}.npz\'\n  if result_path.exists() and pred_path.exists():print(\'RESUME_SKIP\',key,flush=True);continue\n  try:\n   tr,va,te=split_indices(y,d,fp,hold,seed)\n   sv,st,meta=fit_method(method,x,y,d,tr,va,te,seed,(2 if mode==\'pilot\' else CFG[\'rounds\']),device)\n   label=y[te];negative=sv[y[va]==0]\n   results=[]\n   for target in CFG[\'target_fprs\']:\n    if target==.001 and len(negative)<CFG[\'minimum_benign_for_001\']:\n     results.append(dict(target_fpr=target,status=\'insufficient_benign_validation\',n_benign_val=len(negative)));continue\n    thr,valfpr=threshold(negative,target)\n    pp=st>=thr;tp=int(np.sum(pp&(label==1)));fp_count=int(np.sum(pp&(label==0)));fn=int(np.sum(~pp&(label==1)));tn=int(np.sum(~pp&(label==0)))\n    results.append(dict(target_fpr=target,status=\'evaluated\',threshold=thr,validation_fpr=valfpr,test_fpr=fp_count/max(1,tn+fp_count),attack_recall=tp/max(1,tp+fn),precision=tp/max(1,tp+fp_count),attack_f1=2*tp/max(1,2*tp+fp_count+fn),tp=tp,fp=fp_count,tn=tn,fn=fn))\n   record=dict(holdout=hold,seed=seed,method=method,mode=mode,source_domains=[z for z in DOMAINS if z!=hold],n_train=len(tr),n_validation=len(va),n_test=len(te),n_target_benign=int(np.sum(label==0)),n_target_attacks=int(np.sum(label==1)),test_sha=digest(te),validation_sha=digest(va),auroc=float(roc_auc_score(label,st)),auprc=float(average_precision_score(label,st)),**meta,operating_points=results)\n   pred_path.parent.mkdir(parents=True,exist_ok=True)\n   np.savez_compressed(pred_path,test_ids=te,test_labels=label,test_scores=st.astype(np.float32),validation_ids=va,validation_labels=y[va],validation_scores=sv.astype(np.float32))\n   result_path.parent.mkdir(parents=True,exist_ok=True);save_json(result_path,record)\n   print(\'TASK_OK\',mode,worker_id,ix,\'/\',len(pending),hold,seed,method,\'AUROC\',round(record[\'auroc\'],4),flush=True)\n  except Exception:\n   msg=traceback.format_exc();errors.append(dict(holdout=hold,seed=seed,method=method,error=msg))\n   print(\'TASK_FAILED\',hold,seed,method,msg[-1100:],flush=True)\n save_json(OUT/f\'{mode}_worker_{worker_id}_failures.json\',errors)\n if errors:raise RuntimeError(f\'{len(errors)} {mode} tasks failed; inspect failures JSON\')\n\ndef gate():\n paths=list((OUT/\'records\').glob(\'pilot__*.json\')) if (OUT/\'records\').exists() else []\n if len(paths)!=len(tasks(True)):raise RuntimeError(f\'PILOT_GATE_INCOMPLETE {len(paths)}/{len(tasks(True))}\')\n data=[json.loads(p.read_text()) for p in paths]\n if any(not np.isfinite(r[\'auroc\']) for r in data):raise RuntimeError(\'PILOT_BAD_AUROC\')\n save_json(OUT/\'pilot_gate.json\',dict(status=\'passed\',model_runs=len(data),note=\'Software feasibility gate only. Pilot scores are not used for model selection or target tuning.\'))\n print(\'PILOT_GATE_PASSED\',len(data),flush=True)\n\ndef aggregate():\n records=[]\n for p in sorted((OUT/\'records\').glob(\'main__*.json\')):records.append(json.loads(p.read_text()))\n expected=len(tasks(False));unique={(r[\'holdout\'],r[\'seed\'],r[\'method\']) for r in records}\n if len(unique)!=expected:raise RuntimeError(f\'INCOMPLETE_MAIN_RUNS {len(unique)}/{expected}\')\n if len(records)!=expected:raise RuntimeError(\'DUPLICATED_MAIN_RUNS\')\n for h in DOMAINS:\n  for s in CFG[\'seeds\']:\n   rs=[r for r in records if r[\'holdout\']==h and r[\'seed\']==s]\n   if len(rs)!=len(METHODS) or len({r[\'test_sha\'] for r in rs})!=1 or len({r[\'validation_sha\'] for r in rs})!=1:raise RuntimeError(\'FIXED_TEST_OR_VALIDATION_BROKEN\')\n table=[];resources=[];clients=[]\n for r in records:\n  head={k:r[k] for k in [\'holdout\',\'seed\',\'method\',\'n_train\',\'n_validation\',\'n_test\',\'test_sha\',\'validation_sha\',\'auroc\',\'auprc\']}\n  for z in r[\'operating_points\']:\n   row={**head,**z};table.append(row)\n   if z[\'status\']==\'evaluated\' and z[\'validation_fpr\']>z[\'target_fpr\']+1e-12:raise RuntimeError(\'VALIDATION_FPR_INVARIANT\')\n  resources.append({**{k:r[k] for k in [\'holdout\',\'seed\',\'method\',\'n_train\',\'n_validation\',\'n_test\']},**{k:r[k] for k in [\'train_seconds\',\'optimizer_steps\',\'communication_bytes\']}})\n  for c in r[\'local_clients\']:clients.append(dict(holdout=r[\'holdout\'],seed=r[\'seed\'],method=r[\'method\'],**c))\n mm=pd.DataFrame(table);mm.to_csv(OUT/\'experiment_results.csv\',index=False);pd.DataFrame(resources).to_csv(OUT/\'communication_cost.csv\',index=False)\n pd.DataFrame(clients).to_csv(OUT/\'local_client_results.csv\',index=False)\n evaluated=mm[mm.status==\'evaluated\'].copy()\n numeric=[\'attack_recall\',\'test_fpr\',\'precision\',\'attack_f1\',\'auroc\',\'auprc\']\n summary=evaluated.groupby([\'holdout\',\'method\',\'target_fpr\'])[numeric].agg([\'mean\',\'std\',\'count\']).reset_index()\n summary.columns=[\'_\'.join(map(str,c)).strip(\'_\') for c in summary.columns]\n summary.to_csv(OUT/\'summary_mean_std.csv\',index=False)\n diffs=[]\n for (hold,seed,target),t in evaluated.groupby([\'holdout\',\'seed\',\'target_fpr\']):\n  indexed=t.set_index(\'method\')\n  for metric in (\'attack_recall\',\'test_fpr\',\'auroc\',\'auprc\'):\n   for method in (\'FedAvgMLP\',\'FedProxMLP\'):\n    for baseline in (\'PooledMLP\',\'LocalMLP\',\'PooledLightGBM\'):\n     if method in indexed.index and baseline in indexed.index:diffs.append(dict(holdout=hold,seed=seed,target_fpr=target,metric=metric,method=method,baseline=baseline,delta=float(indexed.at[method,metric]-indexed.at[baseline,metric])))\n pd.DataFrame(diffs).to_csv(OUT/\'paired_comparisons.csv\',index=False)\n fail=[]\n for f in OUT.glob(\'*_worker_*_failures.json\'):fail.extend(json.loads(f.read_text()))\n if fail:raise RuntimeError(\'RECENT_FAILURES_PRESENT\')\n save_json(OUT/\'run_status.json\',dict(status=\'complete\',expected_main_runs=expected,completed_main_runs=len(records),operating_points=len(mm),evaluated_points=len(evaluated),unavailable_points=len(mm)-len(evaluated),fixed_test_passed=True,heldout_target_excluded=True))\n archive=ROOT/CFG[\'archive\']\n with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n  for f in OUT.rglob(\'*\'):\n   if f.is_file():z.write(f,f.relative_to(ROOT))\n print(\'FINAL_ZIP\',archive,\'bytes\',archive.stat().st_size,\'runs\',expected,flush=True)\n\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'--mode\',choices=[\'prepare\',\'pilot\',\'gate\',\'main\',\'aggregate\'],required=True);p.add_argument(\'--worker\',type=int,default=0);p.add_argument(\'--workers\',type=int,default=2)\n a=p.parse_args()\n if a.mode==\'prepare\':prepare()\n elif a.mode==\'gate\':gate()\n elif a.mode==\'aggregate\':aggregate()\n else:worker(a.mode,a.worker,a.workers)\nif __name__==\'__main__\':main()\n'
(WORK/'engine02.py').write_text(SOURCE,encoding="utf-8")


In [ ]:
SOURCE = '"""FLIDS 10–13: source-locked experimental suite. Run using the companion notebook."""\nimport os\nfor k in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\'): os.environ[k]=\'1\'\nimport json, hashlib, itertools, time, random, zipfile, argparse, shutil, math\nfrom pathlib import Path\nimport numpy as np, pandas as pd\nimport torch\nfrom torch import nn\nfrom sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split\nfrom scipy.stats import binomtest, beta\nfrom lightgbm import LGBMClassifier\nimport engine02 as original\n\nSEEDS=[0,42,3407,123,2026]\nBASE=[\'LogisticRegression\',\'RandomForest\',\'LightGBM\',\'PooledMLP\',\'LocalMLP\',\'FedAvg\',\'FedProx\',\'FedAdam\',\'FedYogi\']\nEXTRA=[\'GroupDRO\',\'CORAL\',\'DANN_Source\',\'SCAFFOLD\',\'FedNova\',\'FedBN_SourceEnsemble\']\nFL=[\'FedAvg\',\'FedProx\',\'FedAdam\',\'FedYogi\',\'SCAFFOLD\',\'FedNova\',\'FedBN_SourceEnsemble\']\nRATES=[.001,.01,.05]\nDEADLINE=float(\'inf\')\nclass PauseRun(Exception):pass\ndef timegate():\n if time.monotonic()>=DEADLINE:raise PauseRun(\'Configured session budget reached; resume from candidate/bundle records.\')\n\n\ndef seed(*a):return int.from_bytes(hashlib.sha256(repr(a).encode()).digest()[:4],\'little\')\ndef sha(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(2**20),b\'\'):h.update(b)\n return h.hexdigest()\ndef save(p,o):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_suffix(p.suffix+\'.tmp\');q.write_text(json.dumps(o,indent=2,default=str));q.replace(p)\ndef frame(p,rows):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);q=p.with_suffix(\'.tmp\');pd.DataFrame(rows).to_csv(q,index=False);q.replace(p)\ndef npwrite(p,**kw):\n p=Path(p);q=p.with_suffix(\'.tmp\');\n with q.open(\'wb\') as f:np.savez_compressed(f,**kw)\n q.replace(p)\ndef rng(*a):return np.random.default_rng(seed(*a))\ndef cut(s,a):\n s=np.sort(np.asarray(s,dtype=np.float64));k=math.floor(a*len(s));return float(np.nextafter(s[len(s)-k-1],np.inf)) if len(s) else np.inf\ndef conformal(s,a):\n s=np.sort(s);k=math.ceil((len(s)+1)*(1-a));return float(np.nextafter(s[k-1],np.inf)) if k<=len(s) else np.inf\ndef tolcut(s,a,delta):\n # Smallest order-statistic threshold whose one-sided binomial tolerance coverage is >= 1-delta.\n from scipy.stats import binom\n s=np.sort(s);n=len(s);k=int(binom.ppf(1-delta,n,1-a))+1\n return float(np.nextafter(s[k-1],np.inf)) if k<=n else np.inf\ndef auc(y,s,a=None):return float(roc_auc_score(y,s,max_fpr=a))\ndef evalop(y,s,t):\n n=s[y==0];p=s[y==1];fp=int((n>=t).sum());tp=int((p>=t).sum());return fp/len(n),tp/len(p),fp,tp\n\ndef dedup(x,y,d,f):\n # Exclude all cross-domain or conflicting-label/family fingerprints, keep one same-domain duplicate.\n keys=np.array([hashlib.blake2b(np.ascontiguousarray(v).tobytes(),digest_size=16).hexdigest() for v in x]);_,inv,cnt=np.unique(keys,return_inverse=True,return_counts=True)\n bad=set();keep=[];seen={}\n for i,k in enumerate(inv):\n  val=(str(d[i]),int(y[i]),str(f[i]))\n  if k in seen:\n   if seen[k][0]!=val:bad.add(k)\n  else:seen[k]=(val,i)\n keep=np.array([i for k,(_,i) in seen.items() if k not in bad],dtype=int);return keep,dict(original_rows=len(x),kept_rows=len(keep),conflicting_or_cross_domain_groups=len(bad),duplicate_groups=int((cnt>1).sum()))\n\ndef load_nf(work,cfg):\n p=work/\'nf_cache.npz\';audit=work/\'nf_audit.json\'\n if p.exists():\n  a=np.load(p,allow_pickle=False);return tuple(a[k] for k in (\'x\',\'y\',\'d\',\'family\'))\n if cfg[\'smoke\']:\n  original.CFG[\'max_rows_per_domain_class\']=100;original.CFG[\'seeds\']=[0];os.environ[\'FLIDS_EXP02_SMOKE\']=\'1\';original.prepare()\n  a=np.load(original.CACHE)\n elif cfg.get(\'exp09_zip\'):\n  with zipfile.ZipFile(cfg[\'exp09_zip\']) as z:\n   a=np.load(__import__(\'io\').BytesIO(z.read(\'reproducibility/flids_exp02_cache.npz\'))); provenance=json.loads(z.read(\'reproducibility/data_preparation/dataset_audit.json\'))\n   save(work/\'original_nf_provenance.json\',provenance)\n   if provenance.get(\'source_rows_scanned\',0)<1_000_000:raise ValueError(\'Refusing synthetic or incomplete Exp09 cache in full run\')\n else:\n  original.prepare();a=np.load(original.CACHE)\n x,y,d,f=[a[k] for k in (\'x\',\'y\',\'d\',\'attack\')];k,info=dedup(x,y,d,f);save(audit,info);npwrite(p,x=x[k],y=y[k],d=d[k],family=f[k]);return x[k],y[k],d[k],f[k]\n\ndef load_edge(work,cfg):\n path=cfg.get(\'edge_csv\');features=cfg.get(\'edge_features\',[])\n if not path or not features:raise ValueError(\'EDGE_CSV and exact audited EDGE_FEATURES (Safe-30 list) required; no guessed columns\')\n label=cfg[\'edge_label\'];family=cfg[\'edge_family\'];assert len(features)==(30 if cfg.get(\'edge_schema\')==\'Safe-30\' else 19) and len(set(features))==len(features) and not {label,family}&set(features)\n if any(any(t in c.lower() for t in [\'ip.src\',\'ip.dst\',\'srcport\',\'dstport\',\'timestamp\',\'frame.time\',\'payload\',\'attack\',\'label\']) for c in features):raise ValueError(\'Unsafe Edge feature names; verify audited list\')\n df=pd.read_csv(path,usecols=features+[label,family],low_memory=False);y=pd.to_numeric(df[label],errors=\'raise\').to_numpy(dtype=int);assert set(y)=={0,1};f=df[family].astype(str).to_numpy(dtype=str);assert len(set(f[y==1]))==14, \'Expected 14 attack families; verify source file\'\n # Seeded capped per-family/class sample, not natural prevalence.\n ix=[]\n for _,g in df.groupby([label,family]):ix.extend(g.sample(min(len(g),cfg[\'edge_cap_per_family\']),random_state=2077).index)\n ix=np.array(ix)\n fields=df.loc[ix,features].apply(pd.to_numeric,errors=\'coerce\').to_numpy(dtype=float)\n invalid=~np.isfinite(fields)\n save(work/\'edge_numeric_audit.json\',{\'features\':features,\'invalid_by_feature\':dict(zip(features,invalid.sum(axis=0).tolist())),\'excluded_rows\':int(invalid.any(axis=1).sum())})\n valid=~invalid.any(axis=1);x=fields[valid];ix=ix[valid];y=y[ix];f=f[ix];d=np.full(len(y),\'Edge-IIoTset\');k,info=dedup(x,y,d,f);assert len(set(f[k][y[k]==1]))==14, \'Attack family lost after deduplication\';save(work/\'edge_audit.json\',info|{\'schema\':cfg.get(\'edge_schema\'),\'features\':features});return x[k],y[k],d[k],f[k]\n\ndef load_external(path,features,domain,label,family):\n df=pd.read_csv(path,usecols=features+[label,family]);y=pd.to_numeric(df[label],errors=\'raise\').to_numpy(dtype=int);assert set(y)=={0,1};f=df[family].astype(str).to_numpy(dtype=str);ix=[]\n for v in (0,1):\n  ids=np.flatnonzero(y==v);ix.extend(rng(domain,v).choice(ids,min(12000,len(ids)),replace=False))\n ix=np.array(ix);return df.iloc[ix][features].apply(pd.to_numeric,errors=\'raise\').to_numpy(),y[ix],np.full(len(ix),domain),f[ix]\n\ndef groupsplit(ids,y,s,frac):\n a,b=train_test_split(np.asarray(ids),test_size=frac,random_state=s,stratify=y[ids]);return np.sort(a),np.sort(b)\ndef split_outer(y,d,f,held,s,track):\n if track==\'edge\':\n  pos=np.flatnonzero((y==1)&(f==held[0]));ben=np.flatnonzero(y==0);remain,ben_test=groupsplit_ben(ben,s,.20);src=np.r_[remain,np.flatnonzero((y==1)&(f!=held[0]))];test=np.r_[ben_test,pos];domains=[str(d[0])]\n else:\n  src=np.flatnonzero(~np.isin(d,held));test=np.flatnonzero(np.isin(d,held));domains=sorted(set(d[src]))\n fit=[];stop=[];cal=[];st=[]\n for dom in domains:\n  ids=src[d[src]==dom];a,b=groupsplit(ids,y,seed(s,dom,\'outer\'),.30);c,e=groupsplit(b,y,seed(s,dom,\'cal\'),2/3);g,h=groupsplit(e,y,seed(s,dom,\'st\'),.5);fit.extend(a);stop.extend(c);cal.extend(g);st.extend(h)\n parts=tuple(np.array(v,dtype=int) for v in (fit,stop,cal,st,test));assert len(np.unique(np.concatenate(parts)))==sum(map(len,parts));assert all(set(y[v])=={0,1} for v in parts);return parts\ndef groupsplit_ben(ids,s,frac):\n v=rng(s,\'ben\').permutation(ids);k=int(len(v)*frac);return v[k:],v[:k]\n\ndef grid(m):\n if m==\'LogisticRegression\':return [dict(C=v) for v in [.01,.1,1,10]]\n if m==\'RandomForest\':return [dict(max_depth=d,min_samples_leaf=l) for d,l in [(12,1),(None,1),(16,5),(None,5)]]\n if m==\'LightGBM\':return [dict(learning_rate=l,num_leaves=n) for l,n in [(.03,31),(.1,31),(.03,63),(.1,63)]]\n lrs=[.001,.003,.01,.03] if m==\'SCAFFOLD\' else [.0003,.001,.003,.01]\n if m==\'FedProx\':return [dict(lr=l,mu=mu) for l,mu in [( .001,.001),(.003,.001),(.01,.001),(.01,.01)]]\n if m in (\'FedAdam\',\'FedYogi\'):return [dict(lr=l,server_lr=v,tau=.01) for l,v in [(.001,.01),(.003,.01),(.003,.05),(.01,.05)]]\n return [dict(lr=l) for l in lrs]\n\ndef partition(ids,y,d,s,scenario=\'native\'):\n r=rng(s,\'allocation\');ids=np.asarray(ids)\n if scenario==\'native\' and len(set(d[ids]))>=2:return [ids[d[ids]==k] for k in sorted(set(d[ids]))]\n K=5 if len(set(d[ids]))==1 else 3\n if scenario==\'native\':\n  buckets=[[] for _ in range(K)]\n  for lab in (0,1):\n   a=r.permutation(ids[y[ids]==lab]);p=r.dirichlet(np.full(K,.5));chunks=np.split(a,(np.cumsum(p)[:-1]*len(a)).astype(int));\n   for j,c in enumerate(chunks):buckets[j].extend(c)\n  return [np.asarray(a,dtype=int) for a in buckets if len(a)]\n if scenario==\'domain\':return [ids[d[ids]==k] for k in sorted(set(d[ids]))]\n buckets=[[] for _ in range(K)]\n # Stratify jointly on domain and binary class, retaining every source row.\n for dom in sorted(set(d[ids])):\n  for lab in (0,1):\n   a=r.permutation(ids[(d[ids]==dom)&(y[ids]==lab)])\n   if scenario.startswith(\'quantity\'):p=np.array([.7,.2,.1]) if K==3 else np.arange(K,0,-1)/sum(range(1,K+1))\n   elif scenario.startswith(\'label\'):p=np.array([.85,.1,.05]) if lab else np.array([.05,.1,.85]);p=p if K==3 else np.roll(np.array([.7,.1,.1,.05,.05]),lab*3)\n   else:p=np.full(K,1/K)\n   for j,c in enumerate(np.split(a,(np.cumsum(p)[:-1]*len(a)).astype(int))):buckets[j].extend(c)\n return [np.asarray(a,dtype=int) for a in buckets if len(a)]\n\ndef score(net,x,ids,device):\n net.eval();out=[]\n with torch.no_grad():\n  for a in np.array_split(ids,max(1,math.ceil(len(ids)/4096))):out.append(torch.sigmoid(net(x[a])).cpu().numpy())\n return np.concatenate(out).astype(np.float64)\n\nclass MLPBN(nn.Module):\n def __init__(self,n):\n  super().__init__();self.net=nn.Sequential(nn.Linear(n,64),nn.BatchNorm1d(64),nn.ReLU(),nn.Linear(64,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Linear(32,1))\n def forward(self,x):return self.net(x).squeeze(-1)\n\ndef fit_bn(X,y,tr,va,clients,cfg,s,opts,participation,t0):\n device=opts[\'device\'];xx=torch.as_tensor(X,device=device);yy=torch.as_tensor(y,dtype=torch.float32,device=device);torch.manual_seed(s);r=rng(s,\'fedbn\');nets=[MLPBN(X.shape[1]).to(device) for _ in clients];init=nets[0].state_dict()\n for n in nets:n.load_state_dict(init)\n shared=[k for k in init if k.startswith((\'net.0.\',\'net.3.\',\'net.6.\'))];state={k:init[k].detach().clone() for k in shared};history=[];steps=comm=0;best=-np.inf;best_state=None;best_round=wait=0\n def pred(ids):return np.mean([score(n,xx,ids,device) for n in nets],axis=0)\n def sync():\n  for n in nets:\n   st=n.state_dict();st.update(state);n.load_state_dict(st)\n for rd in range(1,opts[\'rounds\']+1):\n  sync();picked=r.choice(len(clients),max(1,math.ceil(participation*len(clients))),replace=False);w=np.array([len(clients[i]) for i in picked],float);w/=w.sum();updates=[];losses=[]\n  for i in picked:\n   n=nets[i];n.train();opt=torch.optim.Adam(n.parameters(),lr=cfg[\'lr\']);ids=clients[i]\n   for _ in range(opts[\'local_steps\']):\n    b=r.choice(ids,max(2,min(opts[\'batch\'],len(ids))),replace=len(ids)<2);opt.zero_grad();loss=nn.functional.binary_cross_entropy_with_logits(n(xx[b]),yy[b]);loss.backward();opt.step();losses.append(float(loss.detach()));steps+=1\n   updates.append({k:n.state_dict()[k].detach().clone() for k in shared});comm+=2*sum(v.numel()*v.element_size() for v in state.values())\n  state={k:sum(float(a)*u[k] for a,u in zip(w,updates)) for k in shared};sync();row=dict(round=rd,train_loss=float(np.mean(losses)),steps=steps,bytes=comm)\n  if rd%opts[\'check_every\']==0 or rd==opts[\'rounds\']:\n   v=auc(y[va],pred(va),.05);row[\'source_stop_pauc05\']=v\n   if v>best+1e-5:best=v;best_round=rd;best_state=[{k:v.detach().cpu().clone() for k,v in n.state_dict().items()} for n in nets];wait=0\n   else:wait+=1\n  history.append(row)\n  if rd>=opts[\'min_round\'] and wait>=opts[\'patience\']:break\n for n,st in zip(nets,best_state):n.load_state_dict(st)\n return pred,history,dict(seconds=time.perf_counter()-t0,steps=steps,bytes=comm,best_round=best_round,rounds_run=rd,stop_reason=\'patience\' if rd<opts[\'rounds\'] else \'round_cap\',source_stop_pauc05=best,client_sizes=[len(i) for i in clients],inference=\'source BN ensemble; no target BN fitting\')\n\ndef fit_model(raw,y,d,tr,va,method,cfg,s,opts,scenario=\'native\',participation=1.):\n """Returns fitted predictor, source-only early-stopping history, metadata. Source val only."""\n t0=time.perf_counter();trans=original.transform(raw);sc=StandardScaler().fit(trans[tr]);X=sc.transform(trans).astype(np.float32);X=np.clip(X,-20,20)\n if method==\'FedBN_SourceEnsemble\':return fit_bn(X,y,tr,va,partition(tr,y,d,s,scenario),cfg,s,opts,participation,t0)\n # Pooled source normalization is an explicit shared-statistics protocol, not private secure aggregation.\n if method in (\'LogisticRegression\',\'RandomForest\',\'LightGBM\'):\n  if method==\'LogisticRegression\':m=LogisticRegression(C=cfg[\'C\'],max_iter=2000,random_state=s)\n  elif method==\'RandomForest\':m=RandomForestClassifier(n_estimators=300 if not opts[\'smoke\'] else 8,n_jobs=2,random_state=s,**cfg)\n  else:m=LGBMClassifier(n_estimators=1500 if not opts[\'smoke\'] else 10,verbosity=-1,n_jobs=2,random_state=s,**cfg)\n  if method==\'LightGBM\':\n   import lightgbm as lgb\n   m.fit(X[tr],y[tr],eval_set=[(X[va],y[va])],eval_metric=\'binary_logloss\',callbacks=[lgb.early_stopping(50,verbose=False)])\n  else:m.fit(X[tr],y[tr])\n  return lambda ids:m.predict_proba(X[ids])[:,1].astype(np.float64),[],dict(seconds=time.perf_counter()-t0,steps=None,bytes=0,stop_reason=\'classical_fit\',best_iteration=getattr(m,\'best_iteration_\',None),iterations=getattr(m,\'n_iter_\',None))\n device=opts[\'device\'];xx=torch.as_tensor(X,device=device);yy=torch.as_tensor(y,dtype=torch.float32,device=device);torch.manual_seed(s);r=rng(s,\'train\');model=original.MLP(X.shape[1]).to(device);init={k:v.detach().clone() for k,v in model.state_dict().items()};clients=partition(tr,y,d,s,scenario);steps=0;comm=0;history=[];best=-np.inf;best_state=None;best_round=0;wait=0\n rounds=opts[\'rounds\'];local_steps=opts[\'local_steps\'];check=opts[\'check_every\'];patience=opts[\'patience\'];min_round=opts[\'min_round\'];B=opts[\'batch\'];lr=cfg[\'lr\'];criterion=nn.BCEWithLogitsLoss();\n def draw(ids):return r.choice(ids,min(B,len(ids)),replace=False)\n def local_update(net,opt,ids,ref=None,control=None):\n  b=draw(ids);opt.zero_grad();loss=criterion(net(xx[b]),yy[b]);\n  if method==\'FedProx\' and ref is not None:loss=loss+.5*cfg[\'mu\']*sum((p-q).square().sum() for p,q in zip(net.parameters(),ref))\n  loss.backward();\n  if control is not None:\n   for p,c in zip(net.parameters(),control):p.grad.add_(c)\n  opt.step();return float(loss.detach())\n if method==\'LocalMLP\':\n  nets=[original.MLP(X.shape[1]).to(device) for _ in clients];[n.load_state_dict(init) for n in nets];optim=[torch.optim.Adam(n.parameters(),lr=lr) for n in nets]\n else:nets=[];optim=[]\n if method in (\'PooledMLP\',\'GroupDRO\',\'CORAL\',\'DANN_Source\'):\n  adversary=nn.Linear(32,len(clients)).to(device) if method==\'DANN_Source\' else None\n  opt=torch.optim.Adam(list(model.parameters())+(list(adversary.parameters()) if adversary is not None else []),lr=lr)\n momentum={k:torch.zeros_like(v) for k,v in init.items()};second={k:torch.zeros_like(v) for k,v in init.items()};cglobal=[torch.zeros_like(p) for p in model.parameters()];clocal=[[torch.zeros_like(p) for p in model.parameters()] for _ in clients];groupw=np.ones(len(clients))/len(clients)\n for rd in range(1,rounds+1):\n  losses=[]\n  if method==\'LocalMLP\':\n   for n,o,ids in zip(nets,optim,clients):\n    n.train()\n    for _ in range(local_steps):losses.append(local_update(n,o,ids));steps+=1\n  elif method in (\'PooledMLP\',\'GroupDRO\',\'CORAL\',\'DANN_Source\'):\n   model.train()\n   for _ in range(local_steps*len(clients)):\n    if method==\'PooledMLP\':losses.append(local_update(model,opt,tr));steps+=1;continue\n    opt.zero_grad();b=[draw(i) for i in clients];ls=torch.stack([criterion(model(xx[i]),yy[i]) for i in b]);loss=ls.mean()\n    if method==\'GroupDRO\':\n     groupw*=np.exp(np.clip(.05*ls.detach().cpu().numpy(),-10,10));groupw/=groupw.sum();loss=(ls*torch.tensor(groupw,device=device,dtype=ls.dtype)).sum()\n    if method==\'DANN_Source\':\n     # Gradient reversal: forward identity; reversed encoder gradient, ordinary domain-head gradient.\n     h=torch.cat([model.net[:-1](xx[i]) for i in b]);rev=2*h.detach()-h;labs=torch.cat([torch.full((len(ids),),j,device=device,dtype=torch.long) for j,ids in enumerate(b)]);loss=loss+.1*nn.functional.cross_entropy(adversary(rev),labs)\n    if method==\'CORAL\':\n     # Align hidden representations in pooled source training; no target data.\n     h=[model.net[:-1](xx[i]) for i in b];cov=[(v-v.mean(0)).T@(v-v.mean(0))/max(1,len(v)-1) for v in h];pairs=list(itertools.combinations(cov,2));loss=loss+(.1*torch.stack([(a-b).square().mean() for a,b in pairs]).mean() if pairs else 0.)\n    loss.backward();opt.step();losses.append(float(loss.detach()));steps+=1\n  else:\n   state={k:v.detach().clone() for k,v in model.state_dict().items()};picked=r.choice(len(clients),max(1,math.ceil(participation*len(clients))),replace=False);weights=np.array([len(clients[i]) for i in picked],float);weights/=weights.sum();deltas=[];counts=[];dc=[]\n   for i in picked:\n    ids=clients[i];local=original.MLP(X.shape[1]).to(device);local.load_state_dict(state);local.train();optimizer=torch.optim.SGD(local.parameters(),lr=lr) if method in (\'SCAFFOLD\',\'FedNova\') else torch.optim.Adam(local.parameters(),lr=lr);ref=[p.detach().clone() for p in local.parameters()];nsteps=local_steps if method!=\'FedNova\' else max(1,min(4*local_steps,round(local_steps*len(ids)/np.mean([len(j) for j in clients]))));control=[a-b for a,b in zip(cglobal,clocal[i])] if method==\'SCAFFOLD\' else None\n    for _ in range(nsteps):losses.append(local_update(local,optimizer,ids,ref,control));steps+=1\n    dd={k:v.detach()-state[k] for k,v in local.state_dict().items()};deltas.append(dd);counts.append(nsteps);comm+=2*sum(v.numel()*v.element_size() for v in state.values())\n    if method==\'SCAFFOLD\':\n     new=[old-g+(q-p.detach())/(nsteps*lr) for old,g,q,p in zip(clocal[i],cglobal,ref,local.parameters())];dc.append([a-b for a,b in zip(new,clocal[i])]);clocal[i]=new;comm+=2*sum(v.numel()*v.element_size() for v in cglobal)\n   if method==\'SCAFFOLD\':\n    for j,g in enumerate(cglobal):g.add_(sum(v[j] for v in dc)/len(clients))\n   avg={k:sum(float(w)*a[k] for w,a in zip(weights,deltas)) for k in state}\n   if method==\'FedNova\':avg={k:float(np.dot(weights,counts))*sum(float(w/n)*a[k] for w,n,a in zip(weights,counts,deltas)) for k in state}\n   if method in (\'FedAdam\',\'FedYogi\'):\n    for k,g in avg.items():\n     momentum[k]=.9*momentum[k]+.1*g;second[k]=.99*second[k]+.01*g.square() if method==\'FedAdam\' else second[k]-.01*g.square()*torch.sign(second[k]-g.square());avg[k]=cfg[\'server_lr\']*(momentum[k]/(1-.9**rd))/((second[k]/(1-.99**rd)).clamp_min(0).sqrt()+cfg[\'tau\'])\n   model.load_state_dict({k:state[k]+avg[k] for k in state})\n  if not np.isfinite(losses).all():raise FloatingPointError(\'nonfinite loss\')\n  row=dict(round=rd,train_loss=float(np.mean(losses)),steps=steps,bytes=comm)\n  if rd%check==0 or rd==rounds:\n   pred=np.mean([score(n,xx,va,device) for n in nets],axis=0) if nets else score(model,xx,va,device);v=auc(y[va],pred,.05);row[\'source_stop_pauc05\']=v\n   if v>best+1e-5:\n    best=v;best_round=rd;best_state=[{k:v.detach().cpu().clone() for k,v in n.state_dict().items()} for n in (nets or [model])];wait=0\n   else:wait+=1\n  history.append(row)\n  if rd>=min_round and wait>=patience:break\n for n,st in zip(nets or [model],best_state):n.load_state_dict(st)\n return (lambda ids:np.mean([score(n,xx,ids,device) for n in nets],axis=0)) if nets else (lambda ids:score(model,xx,ids,device)),history,dict(seconds=time.perf_counter()-t0,steps=steps,bytes=comm,best_round=best_round,rounds_run=rd,stop_reason=\'patience\' if rd<rounds else \'round_cap\',source_stop_pauc05=best,client_sizes=[len(i) for i in clients])\n\ndef bundle(work,raw,y,d,f,held,s,track,method,opts,scenario=\'native\',participation=1.,chosen=None,split_override=None):\n timegate();fit,stop,cal,src_test,test=split_override if split_override is not None else split_outer(y,d,f,held,s,track);key=hashlib.sha256(repr((track,held,s,method,scenario,participation)).encode()).hexdigest()[:20];folder=work/\'bundles\'/key;folder.mkdir(parents=True,exist_ok=True)\n identity=dict(track=track,held=list(held),seed=s,method=method,scenario=scenario,participation=participation,split_digests={k:hashlib.sha256(v.tobytes()).hexdigest() for k,v in zip([\'fit\',\'stop\',\'cal\',\'source_test\',\'target\'],(fit,stop,cal,src_test,test))});meta=folder/\'done.json\'\n if track in (\'edge\',\'temporal\',\'external\'):identity[\'input_identity\']=json.loads((work/(track+\'_input_identity.json\')).read_text())\n if meta.exists() and (folder/\'scores.npz\').exists():\n  obj=json.loads(meta.read_text());assert obj[\'identity\']==identity;assert obj[\'score_sha256\']==sha(folder/\'scores.npz\');return obj[\'config\'],folder\n if chosen is None:\n  candidates=grid(method)[:opts[\'trials\']];rows=[]\n  domains=sorted(set(d[fit]));inner=[]\n  if track!=\'edge\' and len(domains)>=2:\n   for h in domains:\n    a=fit[d[fit]!=h];v=stop[d[stop]!=h];b=np.r_[fit[d[fit]==h],stop[d[stop]==h]];inner.append((h,a,v,b))\n  else:\n   if track==\'temporal\':\n    k=int(len(fit)*.75);a,b=fit[:k],fit[k:]\n   else:a,b=groupsplit(fit,y,seed(s,\'inner\'),.25)\n   inner=[(\'source_validation\',a,b,stop)]\n  for j,cfg in enumerate(candidates):\n   for h,a,v,b in inner:\n    p=folder/f\'trial_{j}_{hashlib.sha1(str(h).encode()).hexdigest()[:8]}.json\'\n    if p.exists():r=json.loads(p.read_text())\n    else:\n     timegate()\n     pred,curve,info=fit_model(raw,y,d,a,v,method,cfg,seed(s,h,j*0),opts,scenario,participation);val=pred(b);r=dict(candidate=j,inner_holdout=h,pauc05=auc(y[b],val,.05),pauc01=auc(y[b],val,.01),auroc=auc(y[b],val),config=cfg,**info);frame(p.with_suffix(\'.csv\'),curve);save(p,r)\n    rows.append(r)\n  chosen_id=int(pd.DataFrame(rows).groupby(\'candidate\').pauc05.mean().sort_values(ascending=False,kind=\'stable\').index[0]);chosen=candidates[chosen_id];frame(folder/\'tuning.csv\',rows)\n  if track!=\'edge\' and len(domains)>=2:\n   inner_rows=[]\n   for h,a,v,b in inner:\n    timegate();cc=cal[d[cal]!=h];ee=src_test[d[src_test]==h]\n    pred0,_,_=fit_model(raw,y,d,a,v,method,chosen,seed(s,h,\'inner_control\'),opts,scenario,participation);ss0=pred0(cc);ss1=pred0(ee)\n    for alpha in RATES:\n     n0=ss0[y[cc]==0];n1=ss1[y[ee]==0];min_n=math.ceil(math.log(.05)/math.log(1-alpha));\n     if min(len(n0),len(n1))<min_n and not opts[\'smoke\']:continue\n     t0=cut(n0,alpha);ff,rr,_,_=evalop(y[ee],ss1,t0);inner_rows.append(dict(track=track,held=\' + \'.join(held),seed=s,method=method,inner_holdout=h,alpha=alpha,fpr=ff,recall=rr,calibration_benign=len(n0),source_test_benign=len(n1)))\n   frame(folder/\'inner_source_control.csv\',inner_rows)\n timegate();pred,curve,info=fit_model(raw,y,d,fit,stop,method,chosen,seed(s,\'final\'),opts,scenario,participation)\n # Target predictions are produced only after source-only selection is frozen.\n scores={};parts={\'fit\':fit,\'stop\':stop,\'cal\':cal,\'source_test\':src_test,\'target\':test}\n for role,ids in parts.items():\n  scores.update({role+\'_ids\':ids,role+\'_y\':y[ids],role+\'_d\':d[ids].astype(str),role+\'_family\':f[ids].astype(str),role+\'_scores\':pred(ids)})\n  assert np.isfinite(scores[role+\'_scores\']).all()\n npwrite(folder/\'scores.npz\',**scores);frame(folder/\'learning.csv\',curve);save(meta,dict(identity=identity,config=chosen,score_sha256=sha(folder/\'scores.npz\'),**info));print(\'DONE\',track,held,s,method,scenario,participation,flush=True);return chosen,folder\n\ndef analyze(folder,opts,work):\n if (folder/\'analysis_complete.json\').exists():return\n obj=json.loads((folder/\'done.json\').read_text());a=np.load(folder/\'scores.npz\');i=obj[\'identity\'];base={**i,\'held\':\' + \'.join(i[\'held\'])};base.pop(\'split_digests\');rows=[];policies=[];few=[];prev=[];alerts=[];families=[];audit=[]\n cy,cs,cd=a[\'cal_y\'],a[\'cal_scores\'],a[\'cal_d\'];sy,ss,sd=a[\'source_test_y\'],a[\'source_test_scores\'],a[\'source_test_d\'];ty,ts,td,tf=a[\'target_y\'],a[\'target_scores\'],a[\'target_d\'],a[\'target_family\'];neg=cs[cy==0];domneg=[neg[cd[cy==0]==j] for j in sorted(set(cd[cy==0]))];\n # Source-fitted Platt calibration; calibration and test roles remain separate.\n pl=LogisticRegression(max_iter=1000).fit(cs.reshape(-1,1),cy);pp=pl.predict_proba(ts.reshape(-1,1))[:,1];bins=np.minimum((pp*15).astype(int),14);ece=sum(np.mean(bins==k)*abs(pp[bins==k].mean()-ty[bins==k].mean()) for k in set(bins));\n for dom in sorted(set(td)):\n  sel=td==dom;y=ty[sel];s=ts[sel];n=s[y==0];p=s[y==1];assert len(n) and len(p)\n  common=base|dict(target_domain=dom,auroc=auc(y,s),pauc01=auc(y,s,.01),pauc05=auc(y,s,.05),ap=float(average_precision_score(y,s)),score_inversion=auc(y,s)<.5,unique_scores=len(np.unique(s)),largest_tie_fraction=np.unique(s,return_counts=True)[1].max()/len(s),brier=float(brier_score_loss(ty,pp)),ece=ece)\n  for alpha in RATES:\n   min_n=math.ceil(math.log(.05)/math.log(1-alpha));adequate=len(neg)>=min_n and len(n)>=min_n\n   if not adequate and not opts[\'smoke\']:\n    rows.append(common|dict(alpha=alpha,status=\'insufficient_tail_sample\',cal_n=len(neg),target_benign_n=len(n),min_n=min_n));continue\n   t=cut(neg,alpha);oracle=cut(n,alpha);fpr,rec,fp,tp=evalop(y,s,t);of,orr,_,_=evalop(y,s,oracle);sf,sr,_,_=evalop(sy,ss,t);N=len(n);lo=0 if fp==0 else beta.ppf(.025,fp,N-fp+1);hi=1 if fp==N else beta.ppf(.975,fp+1,N-fp)\n   rows.append(common|dict(alpha=alpha,status=\'evaluated\',threshold=t,target_fpr=fpr,recall=rec,fp=fp,tp=tp,cal_n=len(neg),target_benign_n=N,target_attack_n=len(p),source_test_fpr=sf,source_test_recall=sr,oracle_fpr=of,oracle_recall=orr,oracle_minus_locked_recall=orr-rec,fpr_ratio=fpr/alpha,flow_iid_binom_p=binomtest(fp,N,alpha,alternative=\'greater\').pvalue,fpr_ci_low=lo,fpr_ci_high=hi,nominal_excess=fpr>alpha))\n   # Three correction policies plus a candidate federated risk-aware tolerance rule.\n   # FedRAT prototype uses per-client tolerance bounds, Bonferroni over source clients.\n   # It is a source-side heuristic, not a distribution-free guarantee for unseen domains.\n   rules={\'empirical_pooled\':t,\'pooled_conformal\':conformal(neg,alpha),\'worst_source_conformal\':max(conformal(v,alpha) for v in domneg),\'minimax_empirical\':max(cut(v,alpha) for v in domneg),\'FedRAT_tolerance_prototype\':max(tolcut(v,alpha,.05/len(domneg)) for v in domneg)}\n   for name,thr in rules.items():\n    f0,r0,_,_=evalop(sy,ss,thr);f1,r1,_,_=evalop(y,s,thr);policies.append(common|dict(alpha=alpha,policy=name,threshold=thr,abstain_all=not np.isfinite(thr),source_test_fpr=f0,source_test_recall=r0,target_fpr=f1,recall=r1,quantile_payload_bytes=(8*len(domneg) if name.startswith((\'worst\',\'FedRAT\',\'minimax\')) else None)))\n   for source in sorted(set(sd)):\n    q=sd==source;val=evalop(sy[q],ss[q],t);audit.append(common|dict(alpha=alpha,source_domain=source,source_test_fpr=val[0],source_test_recall=val[1]))\n   if alpha==.01:\n    for family in sorted(set(tf[sel][y==1])):\n     v=s[(y==1)&(tf[sel]==family)];families.append(common|dict(attack_family=family,n=len(v),locked_recall=float((v>=t).mean()),oracle_recall=float((v>=oracle).mean())))\n    for rep in range(5 if not opts[\'smoke\'] else 1):\n     v=rng(i[\'held\'],i[\'seed\'],dom,rep,\'few\').permutation(n);cap=min(2000,len(n)//2);pool=v[:cap];test=v[cap:]\n     for count in [0,100,250,500,1000,2000]:\n      if count>cap:continue\n      thr=t if count==0 else conformal(pool[:count],alpha);few.append(common|dict(repeat=rep,n_target_cal=count,target_fpr=float((test>=thr).mean()),recall=float((p>=thr).mean()),target_test_benign=len(test),target_informed=count>0))\n    for prevalence in [.001,.005,.01,.05,.1]:\n     den=prevalence*rec+(1-prevalence)*fpr;want=round(len(n)*prevalence/(1-prevalence));want=min(want,len(p));sample=rng(i[\'held\'],i[\'seed\'],dom,prevalence,\'prevalence\').choice(p,want,replace=False);ppv=float((sample>=t).sum()/max(1,(n>=t).sum()+(sample>=t).sum()));prev.append(common|dict(prevalence_assumed=prevalence,ppv_formula=prevalence*rec/den if den else 0,alerts_per_100k=100000*den,subsample_attack_n=want,realized_sample_prevalence=want/(want+len(n)),subsample_ppv=ppv))\n    for budget in [50,100,500,1000]:\n     # source-only assumed prevalence weighted tail; deterministic ties, no target selection.\n     pi=.01;unique=np.unique(cs);sortneg=np.sort(cs[cy==0]);sortpos=np.sort(cs[cy==1]);rates=(1-pi)*(1-np.searchsorted(sortneg,unique,side=\'right\')/len(sortneg))+pi*(1-np.searchsorted(sortpos,unique,side=\'right\')/len(sortpos));idx=np.flatnonzero(rates<=budget/100000);j=idx[0] if len(idx) else len(unique)-1;thr=float(np.nextafter(unique[j],np.inf));f0,r0,_,_=evalop(y,s,thr);alerts.append(common|dict(budget=budget,assumed_prevalence=pi,threshold=thr,source_expected_alerts=100000*rates[j],target_expected_alerts=100000*((1-pi)*f0+pi*r0),target_fpr=f0,recall=r0,source_tie_count=int((cs==unique[j]).sum())))\n for name,v in [(\'operating\',rows),(\'policies\',policies),(\'few_shot\',few),(\'prevalence\',prev),(\'alert_budget\',alerts),(\'attack_family\',families),(\'source_control\',audit)]:\n  if v:frame(folder/(name+\'.csv\'),v)\n save(folder/\'analysis_complete.json\',{\'scores_sha256\':obj[\'score_sha256\']})\n\ndef feature_shift(raw,y,d,held,s,work,fit,test):\n if len(set(d[test]))<1:return\n rows=[];X=original.transform(raw)\n for h in sorted(set(d[test])):\n  a=fit[y[fit]==0];b=test[(y[test]==0)&(d[test]==h)];n=min(2000,len(a),len(b));r=rng(held,s,h,\'shift\');ix=np.r_[r.choice(a,n,replace=False),r.choice(b,n,replace=False)];lab=np.r_[np.zeros(n),np.ones(n)];tr,te=train_test_split(np.arange(len(ix)),test_size=.5,stratify=lab,random_state=s);sc=StandardScaler().fit(X[ix[tr]]);model=LogisticRegression(max_iter=1000).fit(sc.transform(X[ix[tr]]),lab[tr]);rows.append(dict(held=\' + \'.join(held),seed=s,target_domain=h,domain_auc=auc(lab[te],model.predict_proba(sc.transform(X[ix[te]]))[:,1])))\n frame(work/\'shift\'/f\'{seed(held,s)}.csv\',rows)\n\ndef export(work,opts,status):\n summary=work/\'summary\';summary.mkdir(exist_ok=True)\n for name in [\'operating\',\'policies\',\'few_shot\',\'prevalence\',\'alert_budget\',\'attack_family\',\'source_control\',\'inner_source_control\',\'tuning\']:\n  paths=list((work/\'bundles\').glob(\'*/\'+name+\'.csv\'));dfs=[pd.read_csv(p).assign(bundle=p.parent.name) for p in paths if p.stat().st_size>1];\n  if not dfs:continue\n  df=pd.concat(dfs,ignore_index=True);df.to_csv(summary/(name+\'.csv\'),index=False)\n  if name==\'operating\':\n   good=df[df.status==\'evaluated\'];cols=[\'recall\',\'target_fpr\',\'auroc\',\'pauc01\',\'pauc05\',\'oracle_recall\',\'fpr_ratio\',\'source_test_fpr\'];keys=[\'track\',\'scenario\',\'participation\',\'method\',\'held\',\'target_domain\',\'alpha\'];g=good.groupby(keys)[cols].agg([\'mean\',\'std\',\'count\']);g.columns=[\'_\'.join(c) for c in g.columns];g.to_csv(summary/\'domain_mean_sd.csv\');\n   # Domain-level summaries with domain bootstrap, descriptive only (four NF sources).\n   groups=[\'track\',\'scenario\',\'participation\',\'method\',\'alpha\'];dm=good.groupby(groups+[\'target_domain\'])[cols].mean().reset_index();out=[]\n   for k,g in dm.groupby(groups):\n    for c in cols:\n     v=g[c].dropna().to_numpy();n=len(v)\n     if not n:continue\n     b=rng(k,c,\'bootstrap\').choice(v,(2000,n),replace=True).mean(axis=1);out.append(dict(zip(groups,k))|dict(metric=c,n_domains=n,mean=v.mean(),sd=v.std(ddof=1) if n>1 else np.nan,domain_bootstrap_low=np.quantile(b,.025) if n>1 else np.nan,domain_bootstrap_high=np.quantile(b,.975) if n>1 else np.nan))\n   frame(summary/\'descriptive_domain_bootstrap.csv\',out)\n   # Two-stage domain/flow bootstrap approximation: conditional binomial resampling within randomly chosen seed runs.\n   # It assumes flow-level independence and is descriptive, not a substitute for capture-cluster inference.\n   two=[]\n   for k,g in good.groupby(groups):\n    ds=[v for _,v in g.groupby(\'target_domain\')];nd=len(ds)\n    if nd<2:continue\n    R=1000;rr=rng(k,\'two_stage\');domain_draw=rr.integers(nd,size=(R,nd));\n    for metric,ncol in [(\'target_fpr\',\'target_benign_n\'),(\'recall\',\'target_attack_n\')]:\n     values=np.empty((R,nd))\n     for j,v in enumerate(ds):\n      ii,jj=np.where(domain_draw==j);chosen=v.iloc[rr.integers(len(v),size=len(ii))];prob=chosen[metric].to_numpy();nn=chosen[ncol].to_numpy(dtype=int);values[ii,jj]=rr.binomial(nn,prob)/nn\n     means=values.mean(1);two.append(dict(zip(groups,k))|dict(metric=metric,n_domains=nd,lower=np.quantile(means,.025),upper=np.quantile(means,.975),assumption=\'conditional iid flow bootstrap; descriptive\'))\n   if two:frame(summary/\'two_stage_descriptive_bootstrap.csv\',two)\n   # Holm correction of flow-level p values within each matched protocol/alpha, not network-level inference.\n   res=good.copy();res[\'flow_iid_p_holm\']=np.nan\n   for _,g in res.groupby(groups):\n    p=g.flow_iid_binom_p.to_numpy();order=np.argsort(p);adjusted=np.empty(len(p));adjusted[order]=np.minimum(1,np.maximum.accumulate(p[order]*np.arange(len(p),0,-1)));res.loc[g.index,\'flow_iid_p_holm\']=adjusted\n   res.to_csv(summary/\'flow_iid_tests.csv\',index=False)\n   # Score equality audit across methods on identical target IDs; exact tie audit is also in operating.csv.\n   equality=[];bundles={}\n   for p in (work/\'bundles\').glob(\'*/done.json\'):\n    m=json.loads(p.read_text());i=m[\'identity\'];key=(i[\'track\'],tuple(i[\'held\']),i[\'seed\'],i[\'scenario\'],i[\'participation\']);bundles.setdefault(key,[]).append((i[\'method\'],p.parent/\'scores.npz\'))\n   for k,v in bundles.items():\n    loaded=[]\n    for ma,pa in v:\n     with np.load(pa) as a:loaded.append((ma,a[\'target_ids\'],a[\'target_scores\']))\n    for (ma,ia,sa),(mb,ib,sb) in itertools.combinations(loaded,2):\n     if np.array_equal(ia,ib):equality.append(dict(track=k[0],held=\' + \'.join(k[1]),seed=k[2],scenario=k[3],participation=k[4],method_a=ma,method_b=mb,identical_scores=bool(np.array_equal(sa,sb)),max_abs_difference=float(np.max(np.abs(sa-sb)))))\n   frame(summary/\'score_equality_audit.csv\',equality)\n shifts=list((work/\'shift\').glob(\'*.csv\')); \n if shifts:pd.concat([pd.read_csv(p) for p in shifts]).to_csv(summary/\'feature_shift.csv\',index=False)\n status[\'completed_model_bundles\']=len(list((work/\'bundles\').glob(\'*/done.json\')));status[\'completed_tuning_trials\']=len(list((work/\'bundles\').glob(\'*/trial_*.json\')));status[\'completed_analysis_bundles\']=len(list((work/\'bundles\').glob(\'*/analysis_complete.json\')));\n metadata=[]\n for p in (work/\'bundles\').glob(\'*/done.json\'):\n  m=json.loads(p.read_text());i=m[\'identity\'];metadata.append({k:v for k,v in i.items() if k!=\'split_digests\'}|{k:v for k,v in m.items() if k not in [\'identity\',\'score_sha256\',\'config\']}|{\'config\':json.dumps(m[\'config\']),\'best_at_round_cap\':m.get(\'best_round\')==opts[\'rounds\'] if \'best_round\' in m else None})\n if metadata:frame(summary/\'convergence_and_cost.csv\',metadata)\n save(work/\'status.json\',status);save(work/\'environment.json\',dict(torch=torch.__version__,numpy=np.__version__,pandas=pd.__version__,cuda=torch.cuda.is_available(),device=opts[\'device\']))\n # Reproducibility archive excludes temporary files and raw input CSVs, includes score arrays, split IDs, configs and code.\n zpath=work.parent/(\'FLIDS_Packages10_13_SOFTWARE_TEST_ONLY.zip\' if opts[\'smoke\'] else \'FLIDS_Packages10_13_Results.zip\');tmp=zpath.with_suffix(\'.tmp\')\n with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n  for p in work.rglob(\'*\'):\n   if p.is_file() and p.suffix not in (\'.tmp\',\'.pyc\') and not p.name.startswith(\'synthetic_nf\'):z.write(p,p.relative_to(work))\n tmp.replace(zpath);return zpath\n\ndef lock_optional(work,name,identity):\n p=work/(name+\'_input_identity.json\')\n if p.exists() and json.loads(p.read_text())!=identity:raise RuntimeError(name+\' input changed; use a fresh work folder for that track.\')\n save(p,identity)\n\ndef load_temporal(work,e):\n features=original.FEATURES;label=e.get(\'label\',\'Label\');family=e.get(\'family\',\'Attack\');tc=e[\'time_col\'];gc=e.get(\'group_col\');cols=features+[label,family,tc]+([gc] if gc else []);assert len(set(cols))==len(cols)\n # Full chronological scan of timestamps and labels only; second pass loads bounded feature sample.\n meta=pd.read_csv(e[\'path\'],usecols=[tc,label]+([gc] if gc else []),low_memory=False)\n times=pd.to_datetime(pd.to_numeric(meta[tc],errors=\'raise\'),unit=e[\'time_unit\'],utc=True,errors=\'raise\') if e.get(\'time_unit\') else pd.to_datetime(meta[tc],utc=True,errors=\'raise\',format=\'mixed\')\n assert times.notna().all() and times.nunique()>=10\n t=times.astype(\'datetime64[ns, UTC]\').astype(\'int64\').to_numpy();y=pd.to_numeric(meta[label],errors=\'raise\').to_numpy();assert set(y)=={0,1}\n order=np.argsort(t,kind=\'stable\');ts=t[order];boundaries=[ts[min(len(ts)-1,int(len(ts)*q))] for q in [.50,.60,.70,.80]];assert len(set(boundaries))==4,\'Insufficient distinct chronological boundaries\'\n block=np.searchsorted(boundaries,t,side=\'right\');ids=[np.flatnonzero(block==j) for j in range(5)];info={\'source_rows\':len(t),\'boundaries_ns\':[int(v) for v in boundaries],\'capture_group\':gc,\'pre_cap_class_counts\':[{str(v):int((y[ix]==v).sum()) for v in (0,1)} for ix in ids]}\n if gc:\n  g=meta[gc].astype(str);gframe=pd.DataFrame({\'group\':g,\'block\':block});bad=gframe.groupby(\'group\').block.transform(\'min\');good=(block==bad.to_numpy());ids=[ix[good[ix]] for ix in ids];info[\'groups_spanning_time_blocks_excluded\']=int(g[~good].nunique())\n assert all(set(y[ix])=={0,1} for ix in ids),\'A predeclared chronological block lacks a class; supply valid timestamped dataset, do not tune boundaries on target labels.\'\n cap=int(e.get(\'cap_per_class_block\',12000));chosen=np.sort(np.concatenate([rng(\'temporal\',j,v).choice(ix[y[ix]==v],min(cap,int((y[ix]==v).sum())),replace=False) for j,ix in enumerate(ids) for v in (0,1)]));selected=[];offset=0\n for chunk in pd.read_csv(e[\'path\'],usecols=features+[label,family],chunksize=100000,low_memory=False):\n  lo=np.searchsorted(chosen,offset);hi=np.searchsorted(chosen,offset+len(chunk));\n  if hi>lo:selected.append(chunk.iloc[chosen[lo:hi]-offset].copy())\n  offset+=len(chunk)\n assert offset==len(t),\'Input row count changed between scans\';df=pd.concat(selected,ignore_index=True);assert len(df)==len(chosen)\n x=df[features].apply(pd.to_numeric,errors=\'coerce\').to_numpy(dtype=float)\n invalid=~np.isfinite(x);valid=~invalid.any(axis=1)\n audit=[{\'block\':j,\'selected\':int(mask.sum()),\'excluded\':int((mask & ~valid).sum()),\'invalid_by_feature\':dict(zip(features,invalid[mask].sum(axis=0).tolist())),\'retained_by_class\':{str(v):int((mask & valid & (y[chosen]==v)).sum()) for v in (0,1)}} for j in range(5) for mask in [block[chosen]==j]]\n save(work/\'temporal_nonfinite_audit.json\',audit)\n df=df.iloc[valid];chosen=chosen[valid];x=x[valid];yy=pd.to_numeric(df[label],errors=\'raise\').to_numpy(dtype=int);assert np.array_equal(yy,y[chosen]);f=df[family].astype(str).to_numpy(dtype=str);d=np.full(len(yy),str(e.get(\'domain\',\'TemporalNetFlow\')));tt=t[chosen];bb=block[chosen];sort=np.argsort(tt,kind=\'stable\');x,yy,d,f,tt,bb=x[sort],yy[sort],d[sort],f[sort],tt[sort],bb[sort];seen=set();keep=[]\n for i,row in enumerate(x):\n  fingerprint=row.tobytes()\n  if fingerprint not in seen:seen.add(fingerprint);keep.append(i)\n k=np.asarray(keep,dtype=int);du={\'later_duplicates_removed\':len(x)-len(k)};x,yy,d,f,tt,bb=x[k],yy[k],d[k],f[k],tt[k],bb[k];sp=tuple(np.flatnonzero(bb==j) for j in range(5))\n assert all(set(yy[ix])=={0,1} for ix in sp),\'Deduplication removed a class from chronological block\';assert all(tt[a].max()<tt[b].min() for a,b in zip(sp,sp[1:]));n=int(len(sp[0])*.75);assert set(yy[sp[0][:n]])=={0,1} and set(yy[sp[0][n:]])=={0,1},\'Chronological inner split lacks both classes\'\n npwrite(work/\'temporal_cache.npz\',x=x,y=yy,d=d,family=f,timestamps_ns=tt,block=bb);save(work/\'temporal_audit.json\',info|du|{\'split_sizes\':[len(ix) for ix in sp],\'capture_independence_established\':False,\'natural_prevalence_preserved\':False});return x,yy,d,f,sp\n\ndef prepare_optional(work,opts,raw,y,d,f):\n # Pre-train full data preparation of supplied optional tracks, not just header checks.\n # Successful cached preparations are content-locked; missing or invalid tracks are reported explicitly.\n out={};status={}\n for name in (\'edge\',\'external\',\'temporal\'):\n  configured={\'edge\':bool(opts.get(\'edge_csv\') and opts.get(\'edge_features\')),\'external\':bool(opts.get(\'external_inputs\')),\'temporal\':bool(opts.get(\'temporal_input\'))}[name]\n  if not configured:status[name]={\'state\':\'blocked_missing_input\'};continue\n  try:\n   if name==\'edge\':\n    identity=dict(sha=sha(opts[\'edge_csv\']),features=opts[\'edge_features\'],schema=opts.get(\'edge_schema\'),label=opts[\'edge_label\'],family=opts[\'edge_family\'],cap=opts[\'edge_cap_per_family\'])\n   elif name==\'external\':identity=[{k:v for k,v in e.items() if k!=\'path\'}|{\'sha\':sha(e[\'path\'])} for e in opts[\'external_inputs\']]\n   else:\n    e=opts[\'temporal_input\'];identity={k:v for k,v in e.items() if k!=\'path\'}|{\'sha\':sha(e[\'path\'])}\n   identity={\'protocol\':\'v3\',\'contract\':identity};cached=work/(name+\'_cache.npz\');lock=work/(name+\'_input_identity.json\');audit=work/(name+\'_audit.json\')\n   if lock.exists() and json.loads(lock.read_text())!=identity:raise RuntimeError(name+\' input changed; start new work directory for this track.\')\n   if cached.exists() and lock.exists() and audit.exists():\n    with np.load(cached,allow_pickle=False) as a:\n     vals=tuple(a[k] for k in (\'x\',\'y\',\'d\',\'family\'))\n     if name==\'temporal\':vals=vals+(tuple(np.flatnonzero(a[\'block\']==j) for j in range(5)),)\n   else:\n    if name==\'edge\':vals=load_edge(work,opts)\n    elif name==\'external\':\n     arrays=[(raw,y,d,f)]\n     for e in opts[\'external_inputs\']:arrays.append(load_external(e[\'path\'],original.FEATURES,e[\'domain\'],e.get(\'label\',\'Label\'),e.get(\'family\',\'Attack\')))\n     xr,yr,dr,fr=[np.concatenate([a[j] for a in arrays]) for j in range(4)];assert np.isfinite(xr).all(),\'Nonfinite external data\';k,info=dedup(xr,yr,dr,fr);vals=(xr[k],yr[k],dr[k],fr[k]);save(audit,info|{\'provenance\':opts[\'external_inputs\']})\n     for dom in set(vals[2]):assert set(vals[1][vals[2]==dom])=={0,1},f\'Class missing in {dom}\'\n    else:vals=load_temporal(work,opts[\'temporal_input\'])\n    if name!=\'temporal\':npwrite(cached,x=vals[0],y=vals[1],d=vals[2].astype(str),family=vals[3].astype(str))\n    save(lock,identity)\n   if name==\'edge\':assert np.isfinite(vals[0]).all() and len(set(vals[3][vals[1]==1]))==14\n   # Validate model-relevant split feasibility for every actual requested family/domain and seed before training.\n   seeds=[0] if opts[\'smoke\'] else SEEDS\n   if name==\'edge\':\n    for h in sorted(set(vals[3][vals[1]==1])):\n     for s in seeds:split_outer(vals[1],vals[2],vals[3],(h,),s,\'edge\')\n   if name==\'external\':\n    for e in opts[\'external_inputs\']:\n     for s in seeds:split_outer(vals[1],vals[2],vals[3],(e[\'domain\'],),s,\'external\')\n   out[name]=vals;status[name]={\'state\':\'prepared\',\'rows\':len(vals[1]),\'features\':vals[0].shape[1]}\n  except Exception as e:status[name]={\'state\':\'blocked_invalid_input\',\'reason\':repr(e)}\n  print(\'PREPARE\',name,status[name],flush=True)\n save(work/\'optional_preparation_status.json\',status)\n if opts.get(\'require_all_inputs\') and any(v[\'state\']!=\'prepared\' for v in status.values()):raise RuntimeError(\'Full run blocked before model training: see optional_preparation_status.json\')\n return out,status\n\ndef main(config):\n global DEADLINE\n opts=json.loads(Path(config).read_text());DEADLINE=time.monotonic()+3600*float(opts.get(\'session_hours\',9));work=Path(opts[\'work\']);work.mkdir(parents=True,exist_ok=True);torch.set_num_threads(1);opts[\'device\']=\'cuda\' if torch.cuda.is_available() and not opts[\'smoke\'] else \'cpu\';seeds=[0] if opts[\'smoke\'] else SEEDS;methods=BASE+EXTRA;status=dict(mode=\'software_test_only\' if opts[\'smoke\'] else \'research\',packages={},notes=[\'All results are exploratory benchmark results. No unseen-domain FPR guarantee. No DP or secure aggregation.\',\'Equal candidate counts, not equal wall-clock time; FedNova and SCAFFOLD use SGD and report different step budgets.\',\'Record-level CIs assume independent flows; only four NF domains; no confirmatory mixed-effects inference.\',\'FedRAT is a tolerance-bound prototype, not established novel or distribution-free under shift. Source-domain groups are not physical clients, and in synthetic allocations they do not correspond to client IDs.\',\'Historical Exp03 and Edge score equality cannot be established without their raw original artifacts; new-run audits only.\',\'FedBN is an explicit source-BN-ensemble adaptation for an unseen site; DANN_Source is a source-only adversarial baseline, not unsupervised target adaptation.\']);\n identity={k:v for k,v in opts.items() if k not in [\'device\',\'work\',\'exp09_zip\',\'edge_csv\',\'external_inputs\',\'stages\',\'session_hours\',\'edge_features\',\'edge_label\',\'edge_family\',\'edge_cap_per_family\',\'temporal_input\',\'edge_schema\',\'nf_content_sha\',\'require_all_inputs\']};identity[\'source_sha\']=sha(Path(__file__));identity[\'input_exp09_sha\']=opts.get(\'nf_content_sha\') if opts.get(\'exp09_zip\') else None\n # Fingerprint optional data contracts separately; paths may change on Kaggle resume, content must match.\n # Optional tracks have independent input locks, so later sessions can attach them without invalidating NF results.\n ident=work/\'identity.json\'\n if ident.exists() and json.loads(ident.read_text())!=identity:raise RuntimeError(\'Run identity changed; use a new work directory or restore original config.\')\n save(ident,identity);save(work/\'config.json\',opts);shutil.copy2(__file__,work/\'master.py\') if Path(__file__).resolve()!=(work/\'master.py\').resolve() else None\n if (Path(__file__).parent/\'engine02.py\').resolve()!=(work/\'engine02.py\').resolve():shutil.copy2(Path(__file__).parent/\'engine02.py\',work/\'engine02.py\')\n raw,y,d,f=load_nf(work,opts);domains=sorted(set(d));assert len(domains)==4\n scenarios=[(\'native\',1.),(\'label\',1.)] if opts[\'smoke\'] else [(\'native\',1.),(\'native\',2/3),(\'mixed\',1.),(\'quantity\',1.),(\'label\',1.),(\'domain\',1.),(\'quantity\',2/3),(\'label\',2/3)]\n tuned={};stages=opts[\'stages\'];optional={};optional_status={};holds=[(h,) for h in domains][:1 if opts[\'smoke\'] else len(domains)]\n try:\n  if 13 in stages and not opts[\'smoke\']:\n   optional,optional_status=prepare_optional(work,opts,raw,y,d,f)\n   status[\'optional_inputs\']=optional_status\n  for package in (10,11,12,13):\n   if package not in stages:status[\'packages\'][str(package)]=\'not_requested\';continue\n   if package in (10,11,12):\n    for held in holds:\n     for s in seeds:\n      parts=split_outer(y,d,f,held,s,\'nf\');\n      if package==11:feature_shift(raw,y,d,held,s,work,parts[0],parts[-1])\n      for method in methods:\n       chosen,folder=bundle(work,raw,y,d,f,held,s,\'nf\',method,opts);tuned[held,s,method]=chosen\n       if package in (11,12):analyze(folder,opts,work)\n       if package==11 and method in FL+[\'LocalMLP\']:\n        for scenario,part in scenarios[1:]:\n         _,p=bundle(work,raw,y,d,f,held,s,\'nf\',method,opts,scenario,part,chosen);analyze(p,opts,work)\n    status[\'packages\'][str(package)]=\'completed_on_nf\'\n   else:\n    # Every held pair is independent of tuning; re-tune using only remaining sources.\n    pairs=list(itertools.combinations(domains,2))[:1 if opts[\'smoke\'] else 6]\n    for held in pairs:\n     for s in seeds:\n      for method in methods:\n       _,p=bundle(work,raw,y,d,f,held,s,\'nf_leave_two\',method,opts);analyze(p,opts,work)\n    edge=optional_status.get(\'edge\',{}).get(\'state\',\'not_supplied\');external=optional_status.get(\'external\',{}).get(\'state\',\'not_supplied\');temporal=optional_status.get(\'temporal\',{}).get(\'state\',\'not_supplied\')\n    if \'edge\' in optional:\n     er,ey,ed,ef=optional[\'edge\'];families=sorted(set(ef[ey==1]));\n     for h in families:\n      for s in seeds:\n       for method in methods:\n        _,p=bundle(work,er,ey,ed,ef,(h,),s,\'edge\',method,opts);analyze(p,opts,work)\n     edge=\'completed_all_14_families_\'+opts.get(\'edge_schema\',\'unknown_schema\')\n    if \'external\' in optional:\n     xr,yr,dr,fr=optional[\'external\']\n     for e in opts[\'external_inputs\']:\n      held=(e[\'domain\'],)\n      for s in seeds:\n       for method in methods:\n        _,p=bundle(work,xr,yr,dr,fr,held,s,\'external\',method,opts);analyze(p,opts,work)\n     external=\'completed_supplied_harmonized_domains\'\n    if \'temporal\' in optional:\n     xr,yr,dr,fr,sp=optional[\'temporal\']\n     for s in seeds:\n      for method in methods:\n       _,p=bundle(work,xr,yr,dr,fr,(\'future\',),s,\'temporal\',method,opts,split_override=sp);analyze(p,opts,work)\n     temporal=\'completed_supplied_chronological_split\'\n    status[\'packages\'][\'13\']=dict(nf_leave_two=\'completed\',edge=edge,external=external,temporal=temporal)\n   save(work/\'status.json\',status);print(\'PACKAGE\',package,status[\'packages\'][str(package)],flush=True)\n  status[\'state\']=\'completed_requested_supported_stages\';status[\'all_requested_data_tracks_completed\']=bool(13 in stages and all(str(status[\'packages\'][\'13\'].get(k,\'\')).startswith(\'completed\') for k in (\'nf_leave_two\',\'edge\',\'external\',\'temporal\')));status[\'all_reviewer_requirements_completed\']=False;status[\'unresolved\']=[\'Historical score artifacts not supplied\',\'Temporal/external/Edge completion depends on supplied data contracts\',\'Only descriptive domain inference; no reliable mixed-effects inference with four NF domains\',\'Algorithm novelty and bibliographic verification require independent review\']\n except PauseRun as e:\n  status[\'state\']=\'paused_for_resume\';status[\'reason\']=str(e);save(work/\'status.json\',status)\n except BaseException as e:\n  status[\'state\']=\'interrupted_or_failed\';status[\'error\']=repr(e);save(work/\'status.json\',status);raise\n finally:\n  # On normal completion or caught error, export partial state; Kaggle hard kill may prevent export.\n  out=export(work,opts,status);print(\'ARCHIVE\',out,flush=True)\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--config\',required=True);main(p.parse_args().config)\n'
(WORK/'master.py').write_text(SOURCE,encoding="utf-8")


In [ ]:
# Preflight: actual Experiment 09 cache validation and all additional input contracts.
NF_FEATURES=['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'FLOW_DURATION_MILLISECONDS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE']
spec=importlib.util.spec_from_file_location('flids_preflight',WORK/'preflight.py')
pre=importlib.util.module_from_spec(spec);spec.loader.exec_module(pre)
if SMOKE_TEST:
    RESOLVED={'exp09_zip':'','nf_content_sha':None};REPORT={'mode':'software_test_only'}
else:
    RESOLVED,REPORT=pre.run_preflight(INPUT,WORK,NF_FEATURES,exp09=EXP09_ZIP,edge_csv=EDGE_CSV,edge_features=EDGE_FEATURES,edge_features_json=EDGE_FEATURES_JSON,allow_new_edge=ALLOW_NEW_EDGE_PROTOCOL25,external_inputs=EXTERNAL_INPUTS,temporal_input=TEMPORAL_INPUT,require_all=REQUIRE_ALL_INPUTS and 13 in STAGES)
    display(FileLink(str(WORK/'input_preflight.json')))
print('INPUT PREFLIGHT FINISHED. No model training has started.')

In [ ]:
# Restore checkpoint archives from both Kaggle layouts. Fail closed on mismatched code.
def resume_candidates():
    if RESUME_ZIP: return [Path(RESUME_ZIP)]
    if not INPUT.exists(): return []
    paths=list(INPUT.rglob('FLIDS_Packages10_13_Results.zip'))
    for p in INPUT.rglob('identity.json'):
        try:
            q=json.loads(p.read_text())
            if 'source_sha' in q and 'input_exp09_sha' in q and 'rounds' in q and (p.parent/'config.json').exists():paths.append(p.parent)
        except (ValueError,OSError):pass
    return sorted(set(paths))

import base64,zlib,tempfile
BASE=json.loads(zlib.decompress(base64.b64decode('...')))
OLD={'smoke': False, 'trials': 4, 'rounds': 150, 'min_round': 30, 'local_steps': 8, 'batch': 256, 'check_every': 5, 'patience': 10, 'source_sha': '9d17d23227aa6517d27f6ec3ae083d2b0cb074676452739295a7eb7f92484d90', 'input_exp09_sha': '3cb34f8a174bb9a9ef84647e2c5e49599b0da3abaad589e73f62c968682f97eb'}
NEW={'smoke': False, 'trials': 4, 'rounds': 150, 'min_round': 30, 'local_steps': 8, 'batch': 256, 'check_every': 5, 'patience': 10, 'source_sha': 'eb9ea6fb7d7260eaa9d95e99e7d9af3b04080012823607c8d1b9ec2ffdaa97fe', 'input_exp09_sha': '3cb34f8a174bb9a9ef84647e2c5e49599b0da3abaad589e73f62c968682f97eb'}
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1048576),b''): h.update(chunk)
    return h.hexdigest()

def verify(root):
    identity=json.loads((root/'identity.json').read_text())
    assert identity in (OLD,NEW), 'Unknown checkpoint identity'
    assert digest(root/'master.py')==identity['source_sha'], 'Checkpoint source mismatch'
    config=json.loads((root/'config.json').read_text())
    assert all(config.get(k)==v for k,v in OLD.items() if k not in ('source_sha','input_exp09_sha'))
    assert config['nf_content_sha']==OLD['input_exp09_sha']
    for name,expected in BASE.items():
        assert digest(root/name)==expected, 'Historical checkpoint mismatch: '+name
    for marker in (root/'bundles').glob('*/done.json'):
        score=json.loads(marker.read_text())['score_sha256']
        assert digest(marker.parent/'scores.npz')==score
        analysis=marker.parent/'analysis_complete.json'
        if analysis.exists(): assert json.loads(analysis.read_text())['scores_sha256']==score
    return identity

if not SMOKE_TEST:
    assert RESOLVED['nf_content_sha']==OLD['input_exp09_sha'], 'Experiment 09 content mismatch'
    if (WORK/'recovery_ready').exists():
        assert verify(WORK)==NEW
    else:
        candidates=resume_candidates()
        if len(candidates)!=1: raise ValueError(f'Select one checkpoint with RESUME_ZIP: {candidates}')
        with tempfile.TemporaryDirectory(dir=WORK.parent) as tmp:
            root=candidates[0]
            if root.is_file():
                with zipfile.ZipFile(root) as archive:
                    for item in archive.infolist():
                        assert (Path(tmp)/item.filename).resolve().is_relative_to(Path(tmp).resolve())
                    archive.extractall(tmp)
                root=Path(tmp)
            previous=verify(root)
            for item in root.iterdir():
                if item.name not in ('bundles','summary','shift','nf_cache.npz','nf_audit.json','original_nf_provenance.json','config.json','status.json','run.log') and not (previous==NEW and item.name.startswith(('edge_','external_','temporal_'))): continue
                dest=WORK/item.name
                if item.is_dir(): shutil.copytree(item,dest,dirs_exist_ok=True)
                else: shutil.copy2(item,dest)
            (WORK/'identity.json').write_text(json.dumps(NEW))
            verify(WORK)
            (WORK/'recovery_ready').touch()
    print('Verified historical checkpoint and Experiment 09 input.')


In [ ]:
# Launch all stages; no fit on PREFLIGHT_ONLY.
if not PREFLIGHT_ONLY:
    config=dict(work=str(WORK),smoke=SMOKE_TEST,stages=STAGES,session_hours=SESSION_HOURS,
        require_all_inputs=REQUIRE_ALL_INPUTS and not SMOKE_TEST,
        edge_csv='',edge_features=[],edge_schema='',edge_label='Attack_label',edge_family='Attack_type',edge_cap_per_family=EDGE_CAP_PER_FAMILY,external_inputs=[],temporal_input=None,
        trials=1 if SMOKE_TEST else 4,rounds=2 if SMOKE_TEST else 150,
        min_round=1 if SMOKE_TEST else 30,local_steps=2 if SMOKE_TEST else 8,batch=64 if SMOKE_TEST else 256,check_every=1 if SMOKE_TEST else 5,patience=2 if SMOKE_TEST else 10)
    config.update(RESOLVED)
    conf=WORK/'launch_config.json';conf.write_text(json.dumps(config,indent=2))
    env=os.environ.copy();env['FLIDS_EXP02_ROOT']=str(WORK/'prepare');env['PYTHONUNBUFFERED']='1'
    if SMOKE_TEST:env['FLIDS_EXP02_SMOKE']='1'
    else:env.pop('FLIDS_EXP02_SMOKE',None)
    cmd=[sys.executable,'-X','faulthandler',str(WORK/'master.py'),'--config',str(conf)]
    with (WORK/'run.log').open('a',encoding='utf-8') as log:
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in proc.stdout:log.write(line);log.flush();print(line,end='')
        rc=proc.wait()
    print('Return code:',rc)
    if rc!=0:raise RuntimeError('Run stopped; inspect run.log, status.json and optional_preparation_status.json. A partial archive may be available.')
else:print('PREFLIGHT_ONLY: skipped training.')

In [ ]:
import pandas as pd
name='FLIDS_Packages10_13_SOFTWARE_TEST_ONLY.zip' if SMOKE_TEST else 'FLIDS_Packages10_13_Results.zip'
result=WORK.parent/name
for filename in ['input_preflight.json','optional_preparation_status.json','status.json']:
    p=WORK/filename
    if p.exists():print(filename);print(p.read_text())
if result.exists() and not PREFLIGHT_ONLY:
    with zipfile.ZipFile(result) as z:assert z.testzip() is None
    print('Archive MiB:',round(result.stat().st_size/2**20,2));display(FileLink(str(result)))
for name in ['convergence_and_cost.csv','domain_mean_sd.csv','descriptive_domain_bootstrap.csv']:
    p=WORK/'summary'/name
    if p.exists():display(pd.read_csv(p).head(12))

### External input contract (example; replace with actual verified data)
Place `FLIDS_external_inputs.json` next to the external CSV in the same Kaggle dataset. Each list entry must specify `path` (relative to JSON or absolute), unique `domain`, `label`, `family`, `schema_verified: true` and `provenance` documenting the compatible extraction. A raw unrelated CSV must not be renamed or padded with invented values. If unavailable, set `REQUIRE_ALL_INPUTS=False` and report the external stage blocked.

### Output interpretation
The NF and additional input audits are persisted in `input_preflight.json`, `optional_preparation_status.json`, and the final archive. `completed_requested_supported_stages` is not equivalent to all reviewer concerns closed. Full training may span several Kaggle sessions. Hard termination can lose the current fit; resume only with saved output and same source contracts. Full-scale training and actual optional datasets are not validated in the authoring environment.